# Multimodal Data

## Load Text & Image data, Genres per Movie

In [ ]:
movie_path = './data/movielens/movies_with_overviews.csv'
rating_path = './data/movielens/ratings_filtered.csv'

In [ ]:
movies = pd.read_csv(movie_path)
i = 0
for row in movies.iterrows():
    movies.at[i,'genres'] = movies.at[i,'genres'].replace('|', ' ')
    i += 1
movies.head()
movies['text'] = movies['title'] + ' ' + movies['genres'] + ' ' + movies['overview']
movies.head()

In [ ]:
null_indices = movies[movies.isnull().any(axis=1)].index.tolist()
null_indices

In [ ]:
nanlist = [637, 3534, 4785, 5001, 7810, 8578]
for n in nanlist:
    movies.loc[movies.index[n], 'overview'] = movies.loc[movies.index[n], 'title']
    movies.loc[movies.index[n], 'text'] = movies.loc[n]['title'] + ' ' + movies.iloc[n]['genres']
    

In [ ]:
movies.isnull().any()

In [ ]:
users_path = './data/movielens/user_persona.csv'
users = pd.read_csv(users_path)
for index, row in users.iterrows():
    users.at[index,'top_genres'] = " ".join(eval(users.at[index,'top_genres']))
    users.at[index,'Personality'] = users.at[index,'Personality'].replace(',', '')
    users.at[index,'text'] = str(users.at[index,'top_genres']) + ' ' + str(users.at[index,'Personality'])
    users.at[index,'img_path'] = './data/movielens/Profiles/%s.png'% users.at[index,'userId']
users.head()

In [ ]:
# Load Genres
movie_genres = movies['genres'].apply(lambda x: x.replace(' ', ','))
item_genres = {key: movie_genres[val].split(',') for key, val in movie_mapping.items() if val in movie_genres.index}
item_genres

## Load embeddings (if done)

In [ ]:
# If the embeddings have been generated and saved already
# Load Item Embeddings
with open('./data/movielens/llama3/item_text_image_embeddings.json', 'r') as json_file:
    data = json.load(json_file)

item_embeddings = {}
for outer_key, inner_dict in data.items():
    outer_key = int(outer_key)
    item_embeddings[outer_key] = {}
    for inner_key, value in inner_dict.items():
        item_embeddings[outer_key][inner_key] = torch.tensor(value)    


In [ ]:
# Load User Embeddings
with open('./data/movielens/llama3/user_text_image_embeddings.json', 'r') as json_file:
    data = json.load(json_file)

user_embeddings = {}
for outer_key, inner_dict in data.items():
    outer_key = int(outer_key)
    user_embeddings[outer_key] = {}
    for inner_key, value in inner_dict.items():
        user_embeddings[outer_key][inner_key] = torch.tensor(value)  

In [ ]:
# If the embeddings have been generated and saved already
# Load User_Item Embeddings
with open('./data/movielens/llama3/clean_user_item_text_image_embeddings.json', 'r') as json_file:
    data = json.load(json_file)

# Initialize user and item embeddings dictionaries
user_embeddings = {}
item_embeddings = {}

# Iterate over the data and split between users and items
for outer_key, inner_dict in data.items():
    outer_key = int(outer_key)
    
    # If the outer_key is less than `n`, it's a user embedding, otherwise it's an item embedding
    if outer_key < num_users:
        user_embeddings[outer_key] = {}
        for inner_key, value in inner_dict.items():
            user_embeddings[outer_key][inner_key] = torch.tensor(value)
    else:
        item_embeddings[outer_key] = {}
        for inner_key, value in inner_dict.items():
            item_embeddings[outer_key][inner_key] = torch.tensor(value)

In [ ]:
# If the embeddings have been generated and saved already
# Load User_Item Embeddings
with open('./data/movielens/roberta/clean_user_item_text_image_embeddings.json', 'r') as json_file:
    data = json.load(json_file)

# Initialize user and item embeddings dictionaries
user_embeddings = {}
item_embeddings = {}

# Iterate over the data and split between users and items
for outer_key, inner_dict in data.items():
    outer_key = int(outer_key)
    
    # If the outer_key is less than `n`, it's a user embedding, otherwise it's an item embedding
    if outer_key < num_users:
        user_embeddings[outer_key] = {}
        for inner_key, value in inner_dict.items():
            user_embeddings[outer_key][inner_key] = torch.tensor(value)
    else:
        item_embeddings[outer_key] = {}
        for inner_key, value in inner_dict.items():
            item_embeddings[outer_key][inner_key] = torch.tensor(value)

In [ ]:
# Separate text and image embeddings
user_item_embeddings = {}

# Add user keys with 'user_' prefix
for k, v in user_embeddings.items():
    user_item_embeddings[f"user_{k}"] = v

# Add item keys with 'item_' prefix
for k, v in item_embeddings.items():
    user_item_embeddings[f"item_{k}"] = v
     
# Making 1 dict for the text embeddings
text_embeddings = {k: v['text_embedding'] for k, v in user_item_embeddings .items()}

text_embeddings = torch.stack(list(text_embeddings.values()))

# Making 1 dict for the image embeddings
image_embeddings = {k: v['image_embedding'] for k, v in user_item_embeddings.items()}

image_embeddings = torch.stack(list(image_embeddings.values()))
image_embeddings = image_embeddings.squeeze(1)

## Generate embeddings

### Text Embeddings

In [ ]:
#Pre-req for text embeddings
!pip install ollama
!ollama pull llama3
!pip install llama-index==0.10.32
!pip install langchain
!pip install langchain_community


In [ ]:
from langchain_community.embeddings import OllamaEmbeddings

ollama_emb = OllamaEmbeddings(
    model="llama3",
)

In [ ]:
from transformers import RobertaModel, RobertaTokenizer

# Load the tokenizer and model
text_model_name = 'roberta-base'
tokenizer = RobertaTokenizer.from_pretrained(text_model_name)
text_model = RobertaModel.from_pretrained(text_model_name)

In [ ]:
def generate_text_embedding(text):
 
    # Get the output from the model
    try:
        output = text_model(**encoded_input)
    except TypeError as e:
        print(f"TypeError: {str(e)}")

    # Extract embeddings
    text_emb = output.last_hidden_state.mean(dim=1).squeeze(0)
    return text_emb 

In [ ]:
test_tensor = generate_text_embedding("text")
test_tensor.shape

In [ ]:
test_tensor = generate_text_embedding("text")
test_tensor.shape

In [ ]:
# Placeholder functions for generating embeddings
def generate_text_embedding(text):
    text_emb = torch.tensor(ollama_emb.embed_query(text))
    return text_emb 

### Image embeddings

In [ ]:
#Pre-req for image embeddings
!pip install torchvision
from torchvision import models

In [ ]:
import torch
import random
import torchvision.transforms as transforms
from PIL import Image

def generate_image_embedding(image_path):
    model = models.resnet50(pretrained=True)
    # Remove the final fully connected layer to use the model as a feature extractor
    model = torch.nn.Sequential(*(list(model.children())[:-1]))

    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    try:
        image = Image.open(image_path).convert("RGB")
        image = preprocess(image)
    except:
        # Return a random tensor if image.open fails
        image = torch.randn(3, 224, 224)

    image = image.unsqueeze(0)

    with torch.no_grad():
        embedding = model(image)

    img_emb = torch.flatten(embedding, 1)

    return img_emb

### Mapping

In [ ]:
movie_info = dict()
for i, (old_key, old_value) in enumerate(movie_mapping.items()):
    if i < len(movies):
        movie_info[old_key] = {
            'text': movies.loc[i, 'text'],
            'img_path': movies.loc[i, 'img_path']
        }       


In [ ]:
user_info = dict()
for i, (old_key, old_value) in enumerate(user_mapping.items()):
    if i < len(users):
        user_info[old_key] = {
            'text': users.loc[i, 'text'],
            'img_path': users.loc[i, 'img_path']
        }

In [ ]:
# Combine the user and item mapping dictionaries

combined_info = {}
# Increment keys in dict2 by the maximum key in dict1
increment = max(user_info.keys(), default=0) + 1
adjusted_movie_info = {key + increment: value for key, value in movie_info.items()}

# Combine the dictionaries
combined_info = {**user_info, **adjusted_movie_info}

In [ ]:
from collections import defaultdict
user_item_embeddings = defaultdict(dict)

# Step 2: Generate and track embeddings
for row_id, data in combined_info.items():
    text_embedding = generate_text_embedding(data['text'])
    image_embedding = generate_image_embedding(data['img_path'])
       
    user_item_embeddings[row_id]['text_embedding'] = torch.tensor(text_embedding)
    user_item_embeddings[row_id]['image_embedding'] = torch.tensor(image_embedding)

In [ ]:
from collections import defaultdict
#user_item_embeddings = defaultdict(dict)

# Step 2: Generate and track embeddings
for row_id, data in combined_info.items():
    text_embedding = generate_text_embedding(data['text'])
    #image_embedding = generate_image_embedding(data['img_path'])
       
    user_item_embeddings[row_id]['text_embedding'] = torch.tensor(user_item_embeddings[row_id]['text_embedding'])
    #user_item_embeddings[row_id]['image_embedding'] = torch.tensor(image_embedding)

In [ ]:
len(user_item_embeddings)

In [ ]:
combined_info[1]

In [ ]:
user_item_embeddings[1]['text_embedding'].shape

In [ ]:
# Step 2: Generate and track embeddings
for row_id, data in combined_info.items():
    text_embedding = generate_text_embedding(data['text'])
    #image_embedding = generate_image_embedding(data['img_path'])
       
    user_item_embeddings[row_id]['text_embedding'] = torch.tensor(text_embedding)

In [ ]:
user_item_embeddings_list

In [ ]:
# Download embeddings
def convert_keys_to_strings(d):
    converted = {}
    for key, value in d.items():
        new_key = str(key)  # Convert the key to a string
        if isinstance(value, torch.Tensor):
            converted[new_key] = value.tolist()  # Convert tensor to list
        elif isinstance(value, dict):
            converted[new_key] = convert_keys_to_strings(value)  # Recursively convert nested dicts and their keys
        else:
            converted[new_key] = value  # Keep other types as they are
    return converted

# Convert the mixed dictionary and its keys
user_item_embeddings_list = convert_keys_to_strings(user_item_embeddings)

# Write the converted dictionary to a JSON file
with open('roberta_clean_user_item_text_image_embeddings.json', 'w') as json_file:
    json.dump(user_item_embeddings_list, json_file)

In [ ]:
# Separate text and image embeddings for Roberta

text_embeddings_dict = {k: v['text_embedding'] for k, v in user_item_embeddings.items() if 'text_embedding' in v}
image_embeddings_dict = {k: v['image_embedding'] for k, v in user_item_embeddings.items() if 'image_embedding' in v}

# Ensure all tensors are dense before stacking
def convert_to_dense(tensor):
    if tensor.is_sparse:
        return tensor.to_dense()
    return tensor

# Convert and stack text embeddings
text_embeddings = torch.stack([convert_to_dense(v) for v in text_embeddings_dict.values()])

# Convert and stack image embeddings (if needed)
image_embeddings = torch.stack([convert_to_dense(v) for v in image_embeddings_dict.values()])
image_embeddings = image_embeddings.squeeze(1)

In [ ]:
# Separate text and image embeddings for Llama3
     
# Making 1 dict for the text embeddings
user_item_embeddings_list = {k: v['text_embedding'] for k, v in user_item_embeddings.items() if 'text_embedding' in v}
text_embeddings = torch.stack(list(text_embeddings.values()))

# Making 1 dict for the image embeddings
user_item_embeddings_list = {k: v['image_embedding'] for k, v in user_item_embeddings.items() if 'image_embedding' in v}
image_embeddings = torch.stack(list(image_embeddings.values()))
image_embeddings = image_embeddings.squeeze(1)

# LightGCN Rec Sys


In [ ]:
# import required modules
import random
from tqdm import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import json
import torch
from torch import nn, optim, Tensor

from torch_sparse import SparseTensor, matmul

from torch_geometric.utils import structured_negative_sampling
from torch_geometric.data import download_url, extract_zip
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.typing import Adj

## Dataset

In [ ]:
# load user and movie nodes
def load_node_csv(path, index_col):
    """Loads csv containing node information

    Args:
        path (str): path to csv file
        index_col (str): column name of index column

    Returns:
        dict: mapping of csv row to node id
    """
    df = pd.read_csv(path, index_col=index_col)
    mapping = {index: i for i, index in enumerate(df.index.unique())}
    return mapping


user_mapping = load_node_csv(rating_path, index_col='userId')
movie_mapping = load_node_csv(movie_path, index_col='movieId')

In [ ]:
# load edges between users and movies
def load_edge_csv(path, src_index_col, src_mapping, dst_index_col, dst_mapping, link_index_col, rating_threshold=4):
    """Loads csv containing edges between users and items

    Args:
        path (str): path to csv file
        src_index_col (str): column name of users
        src_mapping (dict): mapping between row number and user id
        dst_index_col (str): column name of items
        dst_mapping (dict): mapping between row number and item id
        link_index_col (str): column name of user item interaction
        rating_threshold (int, optional): Threshold to determine positivity of edge. Defaults to 4.

    Returns:
        torch.Tensor: 2 by N matrix containing the node ids of N user-item edges
    """
    df = pd.read_csv(path)
    edge_index = None
    src = [src_mapping[index] for index in df[src_index_col]]
    dst = [dst_mapping[index] for index in df[dst_index_col]]
    edge_attr = torch.from_numpy(df[link_index_col].values).view(-1, 1).to(torch.long) >= rating_threshold


    edge_index = [[], []]
    for i in range(edge_attr.shape[0]):
        if edge_attr[i]:
            edge_index[0].append(src[i])
            edge_index[1].append(dst[i])

    return torch.tensor(edge_index)


edge_index = load_edge_csv(
    rating_path,
    src_index_col='userId',
    src_mapping=user_mapping,
    dst_index_col='movieId',
    dst_mapping=movie_mapping,
    link_index_col='rating',
    rating_threshold=4,
)

In [ ]:
# split the edges of the graph using a 80/10/10 train/validation/test split
num_users, num_movies = len(user_mapping), len(movie_mapping)
num_interactions = edge_index.shape[1]
all_indices = [i for i in range(num_interactions)]

train_indices, test_indices = train_test_split(
    all_indices, test_size=0.2, random_state=1)
val_indices, test_indices = train_test_split(
    test_indices, test_size=0.5, random_state=1)

train_edge_index = edge_index[:, train_indices]
val_edge_index = edge_index[:, val_indices]
test_edge_index = edge_index[:, test_indices]

In [ ]:
# convert edge indices into Sparse Tensors: https://pytorch-geometric.readthedocs.io/en/latest/notes/sparse_tensor.html
train_sparse_edge_index = SparseTensor(row=train_edge_index[0], col=train_edge_index[1], sparse_sizes=(
    num_users + num_movies, num_users + num_movies))
val_sparse_edge_index = SparseTensor(row=val_edge_index[0], col=val_edge_index[1], sparse_sizes=(
    num_users + num_movies, num_users + num_movies))
test_sparse_edge_index = SparseTensor(row=test_edge_index[0], col=test_edge_index[1], sparse_sizes=(
    num_users + num_movies, num_users + num_movies))

In [ ]:
# function which random samples a mini-batch of positive and negative samples
def sample_mini_batch(batch_size, edge_index):
    """Randomly samples indices of a minibatch given an adjacency matrix

    Args:
        batch_size (int): minibatch size
        edge_index (torch.Tensor): 2 by N list of edges

    Returns:
        tuple: user indices, positive item indices, negative item indices
    """
    edges = structured_negative_sampling(edge_index)
    edges = torch.stack(edges, dim=0)
    indices = random.choices(
        [i for i in range(edges[0].shape[0])], k=batch_size)
    batch = edges[:, indices]
    user_indices, pos_item_indices, neg_item_indices = batch[0], batch[1], batch[2]
    return user_indices, pos_item_indices, neg_item_indices

## Implementing LightGCN



In [ ]:
# defines LightGCN model
class LightGCN(MessagePassing):
  
    def __init__(self, num_users, num_items, embedding_dim=64, K=3, add_self_loops=False):
        super().__init__()
        self.num_users, self.num_items = num_users, num_items
        self.embedding_dim, self.K = embedding_dim, K
        self.add_self_loops = add_self_loops

        self.users_emb = nn.Embedding(
            num_embeddings=self.num_users, embedding_dim=self.embedding_dim) # e_u^0
        self.items_emb = nn.Embedding(
            num_embeddings=self.num_items, embedding_dim=self.embedding_dim) # e_i^0

        nn.init.normal_(self.users_emb.weight, std=0.1)
        nn.init.normal_(self.items_emb.weight, std=0.1)

    def forward(self, edge_index: SparseTensor):
 
        # compute \tilde{A}: symmetrically normalized adjacency matrix
        edge_index_norm = gcn_norm(
            edge_index, add_self_loops=self.add_self_loops)

        emb_0 = torch.cat([self.users_emb.weight, self.items_emb.weight]) # E^0
        embs = [emb_0]
        emb_k = emb_0

        # multi-scale diffusion
        for i in range(self.K):
            emb_k = self.propagate(edge_index_norm, x=emb_k)
            embs.append(emb_k)

        embs = torch.stack(embs, dim=1)
        emb_final = torch.mean(embs, dim=1) # E^K

        users_emb_final, items_emb_final = torch.split(
            emb_final, [self.num_users, self.num_items]) # splits into e_u^K and e_i^K

        # returns e_u^K, e_u^0, e_i^K, e_i^0
        return users_emb_final, self.users_emb.weight, items_emb_final, self.items_emb.weight

    def message(self, x_j: Tensor) -> Tensor:
        return x_j

    def message_and_aggregate(self, adj_t: SparseTensor, x: Tensor) -> Tensor:
        # computes \tilde{A} @ x
        return matmul(adj_t, x)

## Loss Function


In [ ]:
def bpr_loss(users_emb_final, users_emb_0, pos_items_emb_final, pos_items_emb_0, neg_items_emb_final, neg_items_emb_0, lambda_val):
    """Bayesian Personalized Ranking Loss as described in https://arxiv.org/abs/1205.2618

    Args:
        users_emb_final (torch.Tensor): e_u_k
        users_emb_0 (torch.Tensor): e_u_0
        pos_items_emb_final (torch.Tensor): positive e_i_k
        pos_items_emb_0 (torch.Tensor): positive e_i_0
        neg_items_emb_final (torch.Tensor): negative e_i_k
        neg_items_emb_0 (torch.Tensor): negative e_i_0
        lambda_val (float): lambda value for regularization loss term

    Returns:
        torch.Tensor: scalar bpr loss value
    """
    reg_loss = lambda_val * (users_emb_0.norm(2).pow(2) +
                             pos_items_emb_0.norm(2).pow(2) +
                             neg_items_emb_0.norm(2).pow(2)) # L2 loss

    pos_scores = torch.mul(users_emb_final, pos_items_emb_final)
    pos_scores = torch.sum(pos_scores, dim=-1) # predicted scores of positive samples
    neg_scores = torch.mul(users_emb_final, neg_items_emb_final)
    neg_scores = torch.sum(neg_scores, dim=-1) # predicted scores of negative samples

    loss = -torch.mean(torch.nn.functional.softplus(pos_scores - neg_scores)) + reg_loss

    return loss

In [ ]:
import torch.nn.functional as F

def cosine_similarity(x, y):
    return F.cosine_similarity(x, y, dim=-1)

def modified_bpr_loss(users_emb_final, users_emb_0, pos_items_emb_final, pos_items_emb_0, neg_items_emb_final, neg_items_emb_0, lambda_val, alpha=0.5):
    
    reg_loss = lambda_val * (users_emb_0.norm(2).pow(2) +
                             pos_items_emb_0.norm(2).pow(2) +
                             neg_items_emb_0.norm(2).pow(2)) # L2 loss

    pos_scores = torch.mul(users_emb_final, pos_items_emb_final)
    pos_scores = torch.sum(pos_scores, dim=-1) # predicted scores of positive samples
    neg_scores = torch.mul(users_emb_final, neg_items_emb_final)
    neg_scores = torch.sum(neg_scores, dim=-1) # predicted scores of negative samples
    
    # Adding cosine similarity to the scores
    pos_cos_sim = cosine_similarity(users_emb_final, pos_items_emb_final)
    neg_cos_sim = cosine_similarity(users_emb_final, neg_items_emb_final)
    
    # Combine scores with cosine similarity
    pos_scores += pos_cos_sim
    neg_scores += neg_cos_sim


    loss = -torch.mean(torch.nn.functional.softplus(pos_scores - neg_scores)) + reg_loss

    return loss

In [ ]:
def weighted_bpr_loss(users_emb_final, users_emb_0, pos_items_emb_final, pos_items_emb_0, neg_items_emb_final, neg_items_emb_0, lambda_val, graph_weight=0.5, text_image_weight=0.5):
    reg_loss = lambda_val * (users_emb_0.norm(2).pow(2) + pos_items_emb_0.norm(2).pow(2) + neg_items_emb_0.norm(2).pow(2))
    
    # Assume the last parts of the embeddings are text and image embeddings
    # Splitting embeddings into parts
    users_graph, users_text_image = torch.split(users_emb_final, [64, users_emb_final.size(1) - 64], dim=-1)
    pos_graph, pos_text_image = torch.split(pos_items_emb_final, [64, pos_items_emb_final.size(1) - 64], dim=-1)
    neg_graph, neg_text_image = torch.split(neg_items_emb_final, [64, neg_items_emb_final.size(1) - 64], dim=-1)
    
    # Compute scores for graph and text/image parts separately
    pos_scores_graph = (users_graph * pos_graph).sum(dim=-1)
    pos_scores_text_image = (users_text_image * pos_text_image).sum(dim=-1)
    neg_scores_graph = (users_graph * neg_graph).sum(dim=-1)
    neg_scores_text_image = (users_text_image * neg_text_image).sum(dim=-1)
    
    # Weighted sum of scores
    pos_scores = graph_weight * pos_scores_graph + text_image_weight * pos_scores_text_image
    neg_scores = graph_weight * neg_scores_graph + text_image_weight * neg_scores_text_image

    # BPR loss calculation
    loss = -torch.mean(torch.nn.functional.softplus(pos_scores - neg_scores)) + reg_loss

    return loss

In [ ]:
# Enhanced BPR where the cosine similarity is weighted by a learnable parameter
import torch
import torch.nn as nn
import torch.nn.functional as F

class EnhancedBPR(nn.Module):
    def __init__(self, lambda_val, alpha_init=0.5):
        super(EnhancedBPR, self).__init__()
        # lambda value for regularization
        self.lambda_val = lambda_val
        # Learnable weight for balancing cosine similarity
        self.alpha = nn.Parameter(torch.tensor([alpha_init]))
        
    def forward(self, users_emb_final, users_emb_0, pos_items_emb_final, pos_items_emb_0, neg_items_emb_final, neg_items_emb_0):
        # Regularization loss (L2 norm)
        reg_loss = self.lambda_val * (users_emb_0.norm(2).pow(2) + 
                                      pos_items_emb_0.norm(2).pow(2) + 
                                      neg_items_emb_0.norm(2).pow(2))
        
        # Interaction scores
        pos_scores = (users_emb_final * pos_items_emb_final).sum(dim=-1)
        neg_scores = (users_emb_final * neg_items_emb_final).sum(dim=-1)
        
        # Cosine similarities
        pos_cos_sim = F.cosine_similarity(users_emb_final, pos_items_emb_final, dim=-1)
        neg_cos_sim = F.cosine_similarity(users_emb_final, neg_items_emb_final, dim=-1)
        
        # Weighted combination of scores and similarities
        pos_total = pos_scores + self.alpha * pos_cos_sim
        neg_total = neg_scores + self.alpha * neg_cos_sim
        
        # BPR loss calculation
        loss = -torch.mean(F.softplus(pos_total - neg_total)) + reg_loss
        return loss


In [ ]:
model = EnhancedBPR(lambda_val=0.01)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 1000

# Example training loop
for epoch in range(num_epochs):
    optimizer.zero_grad()
    loss = model(users_emb_final, users_emb_0, pos_items_emb_final, pos_items_emb_0, neg_items_emb_final, neg_items_emb_0)
    loss.backward(retain_graph=True)
    optimizer.step()

    print(f"Epoch {epoch}, Loss: {loss.item()}")


## Evaluation Metrics


**Normalized Dicounted Cumulative Gain (NDCG)** 

In [ ]:
# helper function to get N_u
def get_user_positive_items(edge_index):
    """Generates dictionary of positive items for each user

    Args:
        edge_index (torch.Tensor): 2 by N list of edges

    Returns:
        dict: dictionary of positive items for each user
    """
    user_pos_items = {}
    for i in range(edge_index.shape[1]):
        user = edge_index[0][i].item()
        item = edge_index[1][i].item()
        if user not in user_pos_items:
            user_pos_items[user] = []
        user_pos_items[user].append(item)
    return user_pos_items

In [ ]:
# computes recall@K and precision@K
def RecallPrecision_ATk(groundTruth, r, k):
    """Computers recall @ k and precision @ k

    Args:
        groundTruth (list): list of lists containing highly rated items of each user
        r (list): list of lists indicating whether each top k item recommended to each user
            is a top k ground truth item or not
        k (intg): determines the top k items to compute precision and recall on

    Returns:
        tuple: recall @ k, precision @ k
    """
    num_correct_pred = torch.sum(r, dim=-1)  # number of correctly predicted items per user
    # number of items liked by each user in the test set
    user_num_liked = torch.Tensor([len(groundTruth[i])
                                  for i in range(len(groundTruth))])
    recall = torch.mean(num_correct_pred / user_num_liked)
    precision = torch.mean(num_correct_pred) / k
    return recall.item(), precision.item()

In [ ]:
# computes NDCG@K
def NDCGatK_r(groundTruth, r, k):
    """Computes Normalized Discounted Cumulative Gain (NDCG) @ k

    Args:
        groundTruth (list): list of lists containing highly rated items of each user
        r (list): list of lists indicating whether each top k item recommended to each user
            is a top k ground truth item or not
        k (int): determines the top k items to compute ndcg on

    Returns:
        float: ndcg @ k
    """
    assert len(r) == len(groundTruth)

    test_matrix = torch.zeros((len(r), k))

    for i, items in enumerate(groundTruth):
        length = min(len(items), k)
        test_matrix[i, :length] = 1
    max_r = test_matrix
    idcg = torch.sum(max_r * 1. / torch.log2(torch.arange(2, k + 2)), axis=1)
    dcg = r * (1. / torch.log2(torch.arange(2, k + 2)))
    dcg = torch.sum(dcg, axis=1)
    idcg[idcg == 0.] = 1.
    ndcg = dcg / idcg
    ndcg[torch.isnan(ndcg)] = 0.
    return torch.mean(ndcg).item()

In [ ]:
def Compute_Item_Distribution(top_K_items):
    """
    Compute D from the top-k recommended items for each user.
    
    Parameters:
    top_K_items(list): list of lists containing highly rated items of each user
    
    Returns:
    dict: D dictionary where keys are items and values are the number of users for whom the item has been recommended.
    """
    # Initialize dictionary to store item counts
    D = {}
    
    # Iterate through each user and their recommended items
    for items in top_K_items:
        for item in items:
            if item not in D:
                D[item] = 0
            D[item] += 1
    
    return D


In [ ]:
def Compute_SRDP_Prob(top_K_items, k):
    """
    Compute P_i(u) and P_i(U) from the top-k recommended items for each user.
    
    Parameters:
    top_K_items(list): list of lists containing highly rated items of each user
    
    Returns:
    dict: P_u dictionary where keys are (user, item) pairs and values are the probabilities P_i(u).
    dict: P_U dictionary where keys are items and values are the probabilities P_i(U).
    """
    # Initialize dictionaries to store probabilities
    P_u = {}
    P_U = {}
    item_counts = Compute_Item_Distribution(top_K_items)
    
    # Total number of users
    total_users = len(top_K_items)
   
    try:# Compute P_i(u)
        for user in range(total_users):
            items = top_K_items[user]
            nrec = min(len(items), k)
            for item in items:
                P_u[(user, item)] = 1 / nrec
    except KeyError as e:
        # Handle the KeyError
        print(f"KeyError encountered: {e}. Key not found in user_items.")
 
    
    # Compute P_i(U)
    for item, count in item_counts.items():
        P_U[item] = count / total_users
    
    return P_u, P_U


In [ ]:
def SRDPNovatK(top_K_items, k):
    
    P_u, P_U = Compute_SRDP_Prob(top_K_items, k)
    D = Compute_Item_Distribution(top_K_items)
    total_users = len(top_K_items)
    serendipity_sum = 0
    novelty_sum = 0
    
    for u in range(len(top_K_items)):
            user_serendipity = 0
            recommended_items = top_K_items[u]
            nrec = min(len(recommended_items), k)
            
            for i in recommended_items:
                Pi_u = P_u.get((u, i), 0)
                Pi_U = P_U.get(i, 0)
                user_serendipity += max(Pi_u - Pi_U, 0)
                
            serendipity_sum += (1 / nrec) * user_serendipity
            
            user_novelty = 0
            for i in recommended_items:
                D_i = D.get(i, 0)
                novelty_score = -np.log2(D_i / total_users) / nrec
                user_novelty += novelty_score
                
            novelty_sum += user_novelty
    
    serendipity_score = (1 / total_users) * serendipity_sum
    novelty_score = novelty_sum / total_users
    
    return serendipity_score, novelty_score


In [ ]:
def Compute_Diversity_System_Embedding(top_K_items, item_embeddings):
    """
    Compute Max-sum Diversification (MaxDiv) for the top-k recommended items.
    
    Parameters:
    top_K_items (list): list of lists containing recommended items for each user.
    item_embeddings (dict): dictionary where keys are item ids and values are the embeddings (vectors) of each item.
    
    Returns:
    float: MaxDiv score
    """
    # Ensure top_K_items is on CPU and convert to a list of lists
    top_K_items = top_K_items.cpu().tolist()
    
    total_users = len(top_K_items)
    max_div_sum = 0
    
    for user_items in top_K_items:
        # Get embeddings for all items in one user list
        embeddings = torch.stack([item_embeddings[item] for item in user_items])
        
        # Calculate pairwise squared distances for all items at once
        pairwise_distances = torch.cdist(embeddings, embeddings, p=2) ** 2
        
        # Sum the distances, ignoring the diagonal (distance of an item to itself)
        user_diversity = pairwise_distances.sum() - torch.diag(pairwise_distances).sum()
        
        max_div_sum += user_diversity
    
    max_div_score = max_div_sum / total_users
    return max_div_score.item()


In [ ]:
def Compute_Diversity_System_Genre(top_K_items, item_genres):
    """
    Compute Max-sum Diversification (MaxDiv) based only on genres for the top-k recommended items.
    
    Parameters:
    top_K_items (list): list of lists containing recommended items for each user.
    item_genres (dict): dictionary where keys are item ids and values are sets or lists of genres.
    
    Returns:
    float: MaxDiv score based on genre dissimilarity
    """
    # Ensure top_K_items is on CPU and convert to a list of lists
    top_K_items = top_K_items.cpu().tolist()
    
    total_users = len(top_K_items)
    max_div_sum = 0
    
    for user_items in top_K_items:
        user_diversity = 0
        num_items = len(user_items)
        
        if num_items < 2:
            continue  # Skip if the user has fewer than 2 items
        
        # Calculate genre-based dissimilarity
        for i in range(num_items):
            for j in range(i + 1, num_items):
                item_id_i = user_items[i]
                item_id_j = user_items[j]
                
                if item_id_i not in item_genres or item_id_j not in item_genres:
                    continue  # Skip if the item ID is not in the dictionary
                
                genres_i = set(item_genres[item_id_i])
                genres_j = set(item_genres[item_id_j])
                
                # Jaccard dissimilarity based on genre overlap
                intersection = genres_i.intersection(genres_j)
                union = genres_i.union(genres_j)
                genre_dissimilarity = 1 - (len(intersection) / len(union) if len(union) > 0 else 0)
                
                user_diversity += genre_dissimilarity
        
        max_div_sum += user_diversity
    
    max_div_score = max_div_sum / total_users if total_users > 0 else 0
    return max_div_score


In [ ]:
def Compute_Diversity_Personal_Embedding(top_K_items, item_embeddings):
    """
    Compute Intra-List Similarity (ILS) for the top-k recommended items.
    
    Parameters:
    top_K_items (list): list of lists containing recommended items for each user.
    item_embeddings (dict): dictionary where keys are item ids and values are the embeddings (vectors) of each item.
    
    Returns:
    float: ILS score
    """
    # Ensure top_K_items is on CPU and convert to a list of lists
    top_K_items = top_K_items.cpu().tolist()
    
    total_users = len(top_K_items)
    ils_sum = 0
    
    for user_items in top_K_items:
        user_similarity = 0
        num_items = len(user_items)
        
        if num_items < 2:
            continue  # Skip if the user has fewer than 2 items
        
        # Get embeddings for the user's recommended items
        embeddings = torch.stack([item_embeddings[item] for item in user_items])
        
        # Normalize embeddings to compute cosine similarity
        normalized_embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        
        # Compute pairwise cosine similarity
        similarity_matrix = torch.matmul(normalized_embeddings, normalized_embeddings.T)
        
        # Sum off-diagonal similarities (i.e., exclude self-similarities)
        similarity_sum = similarity_matrix.sum() - torch.diag(similarity_matrix).sum()
        
        # Average similarity for this user
        user_similarity = similarity_sum / 2
        
        ils_sum += user_similarity
    
    ils_score = ils_sum / total_users if total_users > 0 else 0
    return ils_score.item()

In [ ]:
def Compute_Diversity_Personal_Genre(top_K_items, item_genres):
    """
    Compute Intra-List Similarity (ILS) based only on genres for the top-k recommended items.
    
    Parameters:
    top_K_items (list): list of lists containing recommended items for each user.
    item_genres (dict): dictionary where keys are item ids and values are sets or lists of genres.
    
    Returns:
    float: ILS score based on genre similarity
    """
    # Ensure top_K_items is on CPU and convert to a list of lists
    top_K_items = top_K_items.cpu().tolist()
    
    total_users = len(top_K_items)
    ils_sum = 0
    
    for user_items in top_K_items:
        user_similarity = 0
        num_items = len(user_items)
        
        if num_items < 2:
            continue  # Skip if the user has fewer than 2 items
        
        # Calculate genre-based similarity
        for i in range(num_items):
            for j in range(i + 1, num_items):
                item_id_i = user_items[i]
                item_id_j = user_items[j]
                
                if item_id_i not in item_genres or item_id_j not in item_genres:
                    continue  # Skip if the item ID is not in the dictionary
                
                genres_i = set(item_genres[item_id_i])
                genres_j = set(item_genres[item_id_j])
                
                # Jaccard similarity based on genre overlap
                intersection = genres_i.intersection(genres_j)
                union = genres_i.union(genres_j)
                genre_similarity = len(intersection) / len(union) if len(union) > 0 else 0
                
                user_similarity += genre_similarity
        
        ils_sum += user_similarity / 2
    
    ils_score = ils_sum / total_users if total_users > 0 else 0
    return ils_score

In [ ]:
def Compute_Diversity_Coverage_Item(top_K_items, total_items):
    """
    Compute the coverage metric for the top-k recommended items.
    
    Parameters:
    top_K_items (list): list of lists containing recommended items for each user.
    total_items (int): total number of items in the entire catalog.
    
    Returns:
    float: Coverage score
    """
    # Ensure top_K_items is on CPU and convert to a list of lists
    top_K_items = top_K_items.cpu().tolist()
    
    recommended_items = set()
    
    for user_items in top_K_items:
        recommended_items.update(user_items)
    #print(recommended_items)
    coverage_score = len(recommended_items) / total_items
    return coverage_score


In [ ]:
import torch

def Compute_Diversity_Coverage_Genre(top_K_items, item_genres, total_genres):
    """
    Compute the genre-based coverage for the top-k recommended items from tensor input.
    
    Parameters:
    top_K_items (tensor): Tensor of size [num_users, num_recs] containing recommended item indices for each user.
    item_genres (dict): Dictionary where keys are item indices (as integers) and values are sets or lists of genres.
    total_genres (set): Set of all genres in the entire catalog.
    
    Returns:
    float: Genre coverage score, representing the percentage of unique genres recommended.
    """
    recommended_genres = set()
    
    # Ensure top_K_items is on CPU and convert to a list of lists
    top_K_items = top_K_items.cpu().tolist()
    
    # Calculate genre coverage
    for user_items in top_K_items:
        for item in user_items:
            genres = item_genres.get(item, [])
            recommended_genres.update(genres)  # Add genres for each recommended item
           
    #print(recommended_genres)
    # Calculate the genre coverage score
    genre_coverage_score = len(recommended_genres) / len(total_genres) if total_genres else 0
    
    return genre_coverage_score

### Testing for New Metrics

In [ ]:
    user_embedding = model.users_emb.weight
    item_embedding = model.items_emb.weight

    # get ratings between every user and item - shape is num users x num movies
    rating = torch.matmul(user_embedding, item_embedding.T)

    # get the top k recommended items for each user
    _, top_K_items = torch.topk(rating, k=20)

In [ ]:
Compute_Diversity_System_Embedding(top_K_items, item_embedding)

In [ ]:
Compute_Diversity_System_Genre(top_K_items, item_genres)

In [ ]:
Compute_Diversity_Personal_Embedding(top_K_items, item_embedding)

In [ ]:
Compute_Diversity_Personal_Genre(top_K_items, item_genres)

In [ ]:
Compute_Diversity_Coverage_Item(top_K_items, num_movies)

In [ ]:
Compute_Diversity_Coverage_Genre(top_K_items, item_genres, unique_genres)

### Wrapper Function

In [ ]:
# wrapper function to get evaluation metrics
def get_metrics(model, edge_index, exclude_edge_indices, k):
    """Computes the evaluation metrics: recall, precision, and ndcg @ k

    Args:
        model (LighGCN): lightgcn model
        edge_index (torch.Tensor): 2 by N list of edges for split to evaluate
        exclude_edge_indices ([type]): 2 by N list of edges for split to discount from evaluation
        k (int): determines the top k items to compute metrics on

    Returns:
        tuple: recall @ k, precision @ k, ndcg @ k
    """
    user_embedding = model.users_emb.weight
    item_embedding = model.items_emb.weight

    # get ratings between every user and item - shape is num users x num movies
    rating = torch.matmul(user_embedding, item_embedding.T)

    for exclude_edge_index in exclude_edge_indices:
        # gets all the positive items for each user from the edge index
        user_pos_items = get_user_positive_items(exclude_edge_index)
        # get coordinates of all edges to exclude
        exclude_users = []
        exclude_items = []
        for user, items in user_pos_items.items():
            exclude_users.extend([user] * len(items))
            exclude_items.extend(items)

        # set ratings of excluded edges to large negative value
        rating[exclude_users, exclude_items] = -(1 << 10)

    # get the top k recommended items for each user
    _, top_K_items = torch.topk(rating, k=k)

    # get all unique users in evaluated split
    users = edge_index[0].unique()

    test_user_pos_items = get_user_positive_items(edge_index)

    # convert test user pos items dictionary into a list
    test_user_pos_items_list = [
        test_user_pos_items[user.item()] for user in users]

    # determine the correctness of topk predictions
    r = []
    for user in users:
        ground_truth_items = test_user_pos_items[user.item()]
        label = list(map(lambda x: x in ground_truth_items, top_K_items[user]))
        r.append(label)
    r = torch.Tensor(np.array(r).astype('float'))

    recall, precision = RecallPrecision_ATk(test_user_pos_items_list, r, k)
    ndcg = NDCGatK_r(test_user_pos_items_list, r, k)
    srdp, nov = SRDPNovatK(top_K_items.tolist(), k)
    div_sys_emb = Compute_Diversity_System_Embedding(top_K_items, item_embedding)
    div_usr_emb = Compute_Diversity_System_Genre(top_K_items, item_genres)
    div_sys_gnr =Compute_Diversity_Personal_Embedding(top_K_items, item_embedding)
    div_usr_gnr = Compute_Diversity_Personal_Genre(top_K_items, item_genres)
    div_cov_itm =Compute_Diversity_Coverage_Item(top_K_items, num_movies)
    div_cov_gnr = Compute_Diversity_Coverage_Genre(top_K_items, item_genres, unique_genres)

    return recall, precision, ndcg, srdp, nov, div_sys_emb, div_usr_emb, div_sys_gnr, div_usr_gnr, div_cov_itm, div_cov_gnr

In [ ]:
# wrapper function to evaluate model
def evaluation(model, edge_index, sparse_edge_index, exclude_edge_indices, k, lambda_val):
    """Evaluates model loss and metrics including recall, precision, ndcg @ k

    Args:
        model (LighGCN): lightgcn model
        edge_index (torch.Tensor): 2 by N list of edges for split to evaluate
        sparse_edge_index (sparseTensor): sparse adjacency matrix for split to evaluate
        exclude_edge_indices ([type]): 2 by N list of edges for split to discount from evaluation
        k (int): determines the top k items to compute metrics on
        lambda_val (float): determines lambda for bpr loss

    Returns:
        tuple: bpr loss, recall @ k, precision @ k, ndcg @ k
    """
    # get embeddings
    users_emb_final, users_emb_0, items_emb_final, items_emb_0 = model.forward(
        sparse_edge_index)
    edges = structured_negative_sampling(
        edge_index, contains_neg_self_loops=False)
    user_indices, pos_item_indices, neg_item_indices = edges[0], edges[1], edges[2]
    users_emb_final, users_emb_0 = users_emb_final[user_indices], users_emb_0[user_indices]
    pos_items_emb_final, pos_items_emb_0 = items_emb_final[
        pos_item_indices], items_emb_0[pos_item_indices]
    neg_items_emb_final, neg_items_emb_0 = items_emb_final[
        neg_item_indices], items_emb_0[neg_item_indices]

    loss = bpr_loss(users_emb_final, users_emb_0, pos_items_emb_final, pos_items_emb_0,
                    neg_items_emb_final, neg_items_emb_0, lambda_val).item()

    recall, precision, ndcg, srdp, nov, div_sys_emb, div_usr_emb, div_sys_gnr, div_usr_gnr, div_cov_itm, div_cov_gnr = get_metrics(
        model, edge_index, exclude_edge_indices, k)

    return loss, recall, precision, ndcg, srdp, nov, div_sys_emb, div_usr_emb, div_sys_gnr, div_usr_gnr, div_cov_itm, div_cov_gnr

## Training

In [ ]:
model = LightGCN(num_users, num_movies)

In [ ]:
# define contants
ITERATIONS = 1000
BATCH_SIZE = 1024
LR = 1e-3
ITERS_PER_EVAL = 50
ITERS_PER_LR_DECAY = 50
K = 20
LAMBDA = 1e-6

In [ ]:
# setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device {device}.")


model = model.to(device)
model.train()

optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.95)

edge_index = edge_index.to(device)
train_edge_index = train_edge_index.to(device)
train_sparse_edge_index = train_sparse_edge_index.to(device)

val_edge_index = val_edge_index.to(device)
val_sparse_edge_index = val_sparse_edge_index.to(device)

In [ ]:
# training loop
train_losses = []
val_losses = []
val_precision = []
val_recall = []
val_ndcg = []
val_srdp = []
val_nov = []
val_div_sys_emb = []
val_div_usr_emb = []
val_div_sys_gnr = []
val_div_usr_gnr = []
val_div_cov_itm = []
val_div_cov_gnr = []

for iter in range(ITERATIONS):
 # forward propagation
    users_emb_final, users_emb_0, items_emb_final, items_emb_0 = model.forward(
        train_sparse_edge_index)

    # mini batching
    user_indices, pos_item_indices, neg_item_indices = sample_mini_batch(
        BATCH_SIZE, train_edge_index)
    user_indices, pos_item_indices, neg_item_indices = user_indices.to(
        device), pos_item_indices.to(device), neg_item_indices.to(device)
    users_emb_final, users_emb_0 = users_emb_final[user_indices], users_emb_0[user_indices]
    pos_items_emb_final, pos_items_emb_0 = items_emb_final[
        pos_item_indices], items_emb_0[pos_item_indices]
    neg_items_emb_final, neg_items_emb_0 = items_emb_final[
        neg_item_indices], items_emb_0[neg_item_indices]

    # loss computation
    train_loss = bpr_loss(users_emb_final, users_emb_0, pos_items_emb_final,
                          pos_items_emb_0, neg_items_emb_final, neg_items_emb_0, LAMBDA)

    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()

    if iter % ITERS_PER_EVAL == 0:
        model.eval()
        val_loss, recall, precision, ndcg, srdp, nov, div_sys_emb, div_usr_emb, div_sys_gnr, div_usr_gnr, div_cov_itm, div_cov_gnr = evaluation(
                model, val_edge_index, val_sparse_edge_index, [train_edge_index], K, LAMBDA)

        print(f"[Iteration {iter}/{ITERATIONS}] train_loss: {round(train_loss.item(), 5)}, val_loss: {round(val_loss, 5)}, val_recall@{K}: {round(recall, 5)}, val_precision@{K}: {round(precision, 5)}, val_ndcg@{K}: {round(ndcg, 5)}, val_srdp@{K}: {round(srdp, 5)}, val_nov@{K}: {round(nov, 5)}, val_div_sys_emb: {round(div_sys_emb, 5)}, val_div_usr_emb: {round(div_usr_emb, 5)}, val_div_sys_gnr: {round(div_sys_gnr, 5)}, val_div_usr_gnr: {round(div_usr_gnr, 5)}, val_div_cov_itm: {round(div_cov_itm, 5)}, val_div_cov_gnr: {round(div_cov_gnr, 5)}")
        train_losses.append(train_loss.item())
        val_losses.append(val_loss)
        val_precision.append(precision)
        val_recall.append(recall)
        val_ndcg.append(ndcg)
        val_srdp.append(srdp)
        val_nov .append(nov)
        val_div_sys_emb.append(div_sys_emb)
        val_div_usr_emb.append(div_usr_emb)
        val_div_sys_gnr.append(div_sys_gnr)
        val_div_usr_gnr.append(div_usr_gnr)
        val_div_cov_itm.append(div_cov_itm)
        val_div_cov_gnr.append(div_cov_gnr)
        model.train()

    if iter % ITERS_PER_LR_DECAY == 0 and iter != 0:
        scheduler.step()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(train_losses))]
plt.plot(iters, train_losses, label='train')
plt.plot(iters, val_losses, label='validation')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('training and validation loss curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_srdp))]
plt.plot(iters, val_srdp, label='srdp')
plt.plot(iters, val_recall, label='recall')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('srdp and recall curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_div_sys_emb))]
plt.plot(iters, val_div_sys_emb, label='div_sys_emb')
plt.plot(iters, val_div_usr_emb, label='div_usr_emb')
plt.plot(iters, val_div_sys_gnr, label='div_sys_gnr')
plt.ylabel('diversity score')
plt.title('diversity curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_div_sys_emb))]
plt.plot(iters, val_div_usr_gnr, label='div_usr_gnr')
plt.plot(iters, val_div_cov_itm, label='div_cov_itm')
plt.plot(iters, val_div_cov_gnr, label='div_cov_gnr')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('diversity curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_srdp))]
plt.plot(iters, val_ndcg, label='ndcg')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('ndcg curve')
plt.legend()
plt.show()

In [ ]:
# evaluate on test set
model.eval()
test_edge_index = test_edge_index.to(device)
test_sparse_edge_index = test_sparse_edge_index.to(device)

test_loss, test_recall, test_precision, test_ndcg, test_srdp, test_nov, test_div_sys_emb, test_div_usr_emb, test_div_sys_gnr, test_div_usr_gnr, test_div_cov_itm, test_div_cov_gnr= evaluation(
            model, test_edge_index, test_sparse_edge_index, [train_edge_index, val_edge_index], K, LAMBDA)

print(f"[test_loss: {round(test_loss, 5)}, test_recall@{K}: {round(test_recall, 5)}, test_precision@{K}: {round(test_precision, 5)}, test_ndcg@{K}: {round(test_ndcg, 5)}, test_srdp@{K}: {round(test_srdp, 5)}, test_nov@{K}: {round(test_nov, 5)}, test_div_sys_emb@{K}: {round(test_div_sys_emb, 5)}, test_div_usr_emb@{K}: {round(test_div_usr_emb, 5)}, test_div_sys_gnr@{K}: {round(test_div_sys_gnr, 5)}, test_div_usr_gnr@{K}: {round(test_div_usr_gnr, 5)}, test_div_cov_itm@{K}: {round(test_div_cov_itm, 5)}, test_div_cov_gnr@{K}: {round(test_div_cov_gnr, 5)}]")
print(f"{round(test_loss, 5)}, {round(test_recall, 5)},{round(test_precision, 5)},{round(test_ndcg, 5)},{round(test_srdp, 5)},{round(test_nov, 5)}, {round(test_div_sys_emb, 5)}, {round(test_div_usr_emb, 5)}, {round(test_div_sys_gnr, 5)}, {round(test_div_usr_gnr, 5)}, {round(test_div_cov_itm, 5)}, {round(test_div_cov_gnr, 5)}")

In [ ]:
from pathlib import Path
data = 'ml-small'
experiment = 'base'

# 1. Create models directory 
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# 2. Create model save path 
MODEL_NAME = "lightgcn_{0}.pth".format(experiment)
MODEL_SAVE_PATH = data / MODEL_PATH / MODEL_NAME

# 3. Save the model state dict 
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model.state_dict(), # only saving the state_dict() only saves the models learned parameters
           f=MODEL_SAVE_PATH) 

## Recommendations for a Given User

In [ ]:
#Load model
from pathlib import Path
data = 'ml-small'
experiment = 'base'

# 1. Create models directory 
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# 2. Create model save path 
MODEL_NAME = "lightgcn_{0}.pth".format(experiment)

model = LightGCN(num_users, num_movies)
model.load_state_dict(torch.load(MODEL_PATH/data/MODEL_NAME)) # Set the model to inference mode

In [ ]:
model.eval()
df = pd.read_csv(movie_path)
movieid_title = pd.Series(df.title.values,index=df.movieId).to_dict()
movieid_genres = df.genres.apply(lambda x: x.replace('|', ','))
movieid_genres = {key: movieid_genres[val].split(',') for key, val in movie_mapping.items() if val in movieid_genres.index}

user_pos_items = get_user_positive_items(edge_index)

In [ ]:
def make_predictions(user_id, num_recs, show_highly_rated=True):
    # Retrieve user index from mapping and user embedding from model
    user_idx = user_mapping[user_id]
    user_embedding = model.users_emb.weight[user_idx]

    # Calculate the scores for all items
    scores = torch.matmul(model.items_emb.weight, user_embedding)

    # Retrieve top scores and their indices, increased to include positively rated items
    top_scores, indices = torch.topk(scores, k=len(user_pos_items[user_idx]) + num_recs)

    # Convert tensor indices to CPU and then to list items
    indices = indices.cpu().tolist()

    # Highly rated movies
    highly_rated_indices = [idx for idx in indices if idx in user_pos_items[user_idx]][:num_recs]
    highly_rated_movie_ids = [inverse_movie_mapping[idx] for idx in highly_rated_indices]
    highly_rated_titles = [movieid_title[movie_id] for movie_id in highly_rated_movie_ids]
    highly_rated_genres = [movieid_genres[movie_id] for movie_id in highly_rated_movie_ids]
    
    if show_highly_rated == True:
        print(f"Here are some movies that user {user_id} rated highly:")
        for title, genre in zip(highly_rated_titles, highly_rated_genres):
            print(f"title: {title}, genres: {genre}")

        print()

    # Suggested new movies
    suggested_indices = [idx for idx in indices if idx not in user_pos_items[user_idx]][:num_recs]
    suggested_movie_ids = [inverse_movie_mapping[idx] for idx in suggested_indices]
    suggested_titles = [movieid_title[movie_id] for movie_id in suggested_movie_ids]
    suggested_genres = [movieid_genres[movie_id] for movie_id in suggested_movie_ids]
    if show_highly_rated == True:
        print(f"Here are some suggested movies for user {user_id}:")
        for title, genre in zip(suggested_titles, suggested_genres):
            print(f"title: {title}, genres: {genre}")
        
    return suggested_genres, suggested_movie_ids

# Helper dictionaries to map movie indices to IDs
inverse_movie_mapping = {v: k for k, v in movie_mapping.items()}


In [ ]:
USER_ID = 11
NUM_RECS = 20

suggested_genres, suggested_movie_ids = make_predictions(USER_ID, NUM_RECS, TRUE)

In [ ]:
# Assess per user, genre diversity and distribution
from collections import Counter

unique_suggested_genres = set([genre for sublist in suggested_genres for genre in sublist])
genre_distribution = Counter([genre for sublist in suggested_genres for genre in sublist])

print(f"No. of genres covered: {len(unique_suggested_genres)} over {len(unique_genres)} : {round(len(unique_suggested_genres) / len(unique_genres) * 100, 2)}%")

# Prepare data for plotting
genres, counts = zip(*genre_distribution.items())

# Create a bar chart
plt.figure(figsize=(10, 8))
plt.bar(genres, counts, color='skyblue')
plt.xlabel('Genres')
plt.ylabel('Frequency')
plt.title('Distribution of Movie Genres for User %s'% USER_ID)
plt.xticks(rotation=45)
plt.show()

In [ ]:
unique_genres

In [ ]:
NUM_RECS = 20

In [ ]:
user_mapping.keys()

In [ ]:
def flatten_list(nested_list):
    for element in nested_list:
        if isinstance(element, list):  # Check if the item is a list
            yield from flatten_list(element)  # Recursive call for items that are lists
        else:
            yield element

In [ ]:
from itertools import chain

# Assess for the entire dataset
suggested_genres, suggested_movies = [], []
for user in user_mapping.keys():
    try:
        user_genres, user_movies = make_predictions(user, NUM_RECS, False)
        suggested_genres.append(user_genres)
        suggested_movies.append(user_movies)
    except:
        continue
    
individual_suggested_genres = list(flatten_list(suggested_genres))
individual_suggested_movies = list(flatten_list(suggested_movies))
unique_suggested_genres = set(individual_suggested_genres)
unique_suggested_movies = set(individual_suggested_movies)

genre_distribution = Counter(individual_suggested_genres)
movie_distribution = Counter(individual_suggested_movies)

print(f"No. of genres covered: {len(unique_suggested_genres)} over {len(unique_genres)} : {round(len(unique_suggested_genres) / len(unique_genres) * 100, 2)}%")
print(f"No. of movies covered: {len(unique_suggested_movies)} over {len(movies)} : {round(len(unique_suggested_movies) / len(movies) * 100, 2)}%")


In [ ]:
# Prepare data for plotting
genres, counts = zip(*genre_distribution.items())

# Create a bar chart
plt.figure(figsize=(10, 8))
plt.bar(genres, counts, color='skyblue')
plt.xlabel('Genres')
plt.ylabel('Frequency')
plt.title('Distribution of Genres for All User')
plt.xticks(rotation=45)
plt.show()

# LightGCN with Text


## Fusion at Propagation

In [ ]:
# Integrating the embeddings at the initialization
import torch.nn.functional as F

class LightGCN_text(MessagePassing):

    def __init__(self, num_users, num_items, text_embeddings, embedding_dim=64, K=3, add_self_loops=False):
        super().__init__()
        self.num_users, self.num_items = num_users, num_items
        self.embedding_dim, self.K = embedding_dim, K
        self.add_self_loops = add_self_loops

        # Load precomputed text and image embeddings
        self.text_embeddings = text_embeddings  # Tensor of size [num_users + num_items, text_emb_dim]
        
        # Embedding layers for users and items
        self.users_emb = nn.Embedding(num_embeddings=self.num_users, embedding_dim=self.embedding_dim)
        self.items_emb = nn.Embedding(num_embeddings=self.num_items, embedding_dim=self.embedding_dim)

        # Initialize embeddings
        nn.init.normal_(self.users_emb.weight, std=0.1)
        nn.init.normal_(self.items_emb.weight, std=0.1)
        
        # Reducing dimensionality to balance
        self.dim_txt_reduction = nn.Linear(self.text_embeddings.shape[1], self.embedding_dim)
       
        # Apply dimensionality reduction without making them trainable
        with torch.no_grad():
            reduced_text = self.dim_txt_reduction(self.text_embeddings)
 
        self.text_embeddings = nn.Parameter(reduced_text, requires_grad=False)
 
    def forward(self, edge_index: SparseTensor):
        # Compute \tilde{A}: symmetrically normalized adjacency matrix
        edge_index_norm = gcn_norm(edge_index, add_self_loops=self.add_self_loops)

        # Trainable user and item embeddings
        emb_trainable_users = self.users_emb.weight
        emb_trainable_items = self.items_emb.weight
        
        # Concatenate trainable embeddings with fixed embeddings for users and items
        emb_users = torch.cat([emb_trainable_users, self.text_embeddings[:num_users]], dim=1)  # Users: trainable + fixed
        emb_items = torch.cat([emb_trainable_items,  self.text_embeddings[num_users:]], dim=1)  # Items: trainable + fixed

        # Initial embeddings, possibly reduced in dimension
        emb_0 = torch.cat([emb_users, emb_items])
        embs = [emb_0]
        emb_k = emb_0

        # Propagate embeddings
        for i in range(self.K):
            emb_k = self.propagate(edge_index_norm, x=emb_k)
            embs.append(emb_k)

        embs = torch.stack(embs, dim=1)
        emb_final = torch.mean(embs, dim=1)

        users_emb_final, items_emb_final = torch.split(emb_final, [self.num_users, self.num_items])

        return users_emb_final, self.users_emb.weight, items_emb_final, self.items_emb.weight
  

    def message(self, x_j: torch.Tensor) -> torch.Tensor:
        return x_j

    def message_and_aggregate(self, adj_t: SparseTensor, x: torch.Tensor) -> torch.Tensor:
        # Computes \tilde{A} @ x
        return matmul(adj_t, x)


## Fusion at Initialization

In [ ]:
# Integrating the embeddings at the initialization
import torch.nn.functional as F
    
class LightGCN_text(MessagePassing):

    def __init__(self, num_users, num_items, text_embeddings, embedding_dim=64, K=3, add_self_loops=False):
        super().__init__()
        self.num_users, self.num_items = num_users, num_items
        self.embedding_dim, self.K = embedding_dim, K
        self.add_self_loops = add_self_loops

        # Load precomputed text and image embeddings
        self.text_embeddings = text_embeddings  # Tensor of size [num_users + num_items, text_emb_dim]
        #self.image_embeddings = image_embeddings  # Tensor of size [num_users + num_items, image_emb_dim]

        # Embedding layers for users and items
        self.users_emb = nn.Embedding(num_embeddings=self.num_users, embedding_dim=self.embedding_dim)
        self.items_emb = nn.Embedding(num_embeddings=self.num_items, embedding_dim=self.embedding_dim)

        # Initialize embeddings
        nn.init.normal_(self.users_emb.weight, std=0.1)
        nn.init.normal_(self.items_emb.weight, std=0.1)
        
        # Reducing dimensionality to balance
        self.dim_txt_reduction = nn.Linear(self.text_embeddings.shape[1], self.embedding_dim)
        self.text_embeddings = nn.Parameter(self.dim_txt_reduction(self.text_embeddings))
               
        # Concatenate graph, text, and image embeddings
        self.users_emb.weight = nn.Parameter(torch.cat([self.users_emb.weight, self.text_embeddings[:self.num_users]], dim=1))
        self.items_emb.weight = nn.Parameter(torch.cat([self.items_emb.weight, self.text_embeddings[self.num_users:]], dim=1))
        
        # Fuse the grap and text embeddings by tensor addition
        # self.users_emb.weight = nn.Parameter(self.users_emb.weight + self.text_embeddings[:self.num_users])
        # self.items_emb.weight = nn.Parameter(self.items_emb.weight + self.text_embeddings[self.num_users:])
        
       # Reducing dimensionality of concatenated embeddings
        #self.dim_usr_reduction = nn.Linear(self.users_emb.weight.shape[1], self.embedding_dim)
        #self.users_emb.weight = nn.Parameter(self.dim_usr_reduction(self.users_emb.weight))
        #self.dim_itm_reduction = nn.Linear(self.items_emb.weight.shape[1], self.embedding_dim)
        #self.items_emb.weight = nn.Parameter(self.dim_itm_reduction(self.items_emb.weight))

    def forward(self, edge_index: SparseTensor):
        # Compute \tilde{A}: symmetrically normalized adjacency matrix
        edge_index_norm = gcn_norm(edge_index, add_self_loops=self.add_self_loops)

        # Initial embeddings, possibly reduced in dimension
        emb_0 = torch.cat([self.users_emb.weight, self.items_emb.weight])
        embs = [emb_0]
        emb_k = emb_0

        # Propagate embeddings
        for i in range(self.K):
            emb_k = self.propagate(edge_index_norm, x=emb_k)
            embs.append(emb_k)

        embs = torch.stack(embs, dim=1)
        emb_final = torch.mean(embs, dim=1)

        users_emb_final, items_emb_final = torch.split(emb_final, [self.num_users, self.num_items])

        return users_emb_final, self.users_emb.weight, items_emb_final, self.items_emb.weight
  

    def message(self, x_j: torch.Tensor) -> torch.Tensor:
        return x_j

    def message_and_aggregate(self, adj_t: SparseTensor, x: torch.Tensor) -> torch.Tensor:
        # Computes \tilde{A} @ x
        return matmul(adj_t, x)


## Training

In [ ]:
model = LightGCN_text(num_users, num_movies, text_embeddings)

In [ ]:
# define contants
ITERATIONS = 1000
BATCH_SIZE = 1024
LR = 1e-3
ITERS_PER_EVAL = 50
ITERS_PER_LR_DECAY = 50
K = 20
LAMBDA = 1e-6

In [ ]:
# setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device {device}.")


model = model.to(device)
model.train()

optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.95)

edge_index = edge_index.to(device)
train_edge_index = train_edge_index.to(device)
train_sparse_edge_index = train_sparse_edge_index.to(device)

val_edge_index = val_edge_index.to(device)
val_sparse_edge_index = val_sparse_edge_index.to(device)

In [ ]:
# training loop
train_losses = []
val_losses = []
val_precision = []
val_recall = []
val_ndcg = []
val_srdp = []
val_nov = []
val_div_sys_emb = []
val_div_usr_emb = []
val_div_sys_gnr = []
val_div_usr_gnr = []
val_div_cov_itm = []
val_div_cov_gnr = []

for iter in range(ITERATIONS):
 # forward propagation
    users_emb_final, users_emb_0, items_emb_final, items_emb_0 = model.forward(
        train_sparse_edge_index)

    # mini batching
    user_indices, pos_item_indices, neg_item_indices = sample_mini_batch(
        BATCH_SIZE, train_edge_index)
    user_indices, pos_item_indices, neg_item_indices = user_indices.to(
        device), pos_item_indices.to(device), neg_item_indices.to(device)
    users_emb_final, users_emb_0 = users_emb_final[user_indices], users_emb_0[user_indices]
    pos_items_emb_final, pos_items_emb_0 = items_emb_final[
        pos_item_indices], items_emb_0[pos_item_indices]
    neg_items_emb_final, neg_items_emb_0 = items_emb_final[
        neg_item_indices], items_emb_0[neg_item_indices]

    # loss computation
    train_loss = bpr_loss(users_emb_final, users_emb_0, pos_items_emb_final,
                          pos_items_emb_0, neg_items_emb_final, neg_items_emb_0, LAMBDA)

    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()

    if iter % ITERS_PER_EVAL == 0:
        model.eval()
        val_loss, recall, precision, ndcg, srdp, nov, div_sys_emb, div_usr_emb, div_sys_gnr, div_usr_gnr, div_cov_itm, div_cov_gnr = evaluation(
                model, val_edge_index, val_sparse_edge_index, [train_edge_index], K, LAMBDA)

        print(f"[Iteration {iter}/{ITERATIONS}] train_loss: {round(train_loss.item(), 5)}, val_loss: {round(val_loss, 5)}, val_recall@{K}: {round(recall, 5)}, val_precision@{K}: {round(precision, 5)}, val_ndcg@{K}: {round(ndcg, 5)}, val_srdp@{K}: {round(srdp, 5)}, val_nov@{K}: {round(nov, 5)}, val_div_sys_emb: {round(div_sys_emb, 5)}, val_div_usr_emb: {round(div_usr_emb, 5)}, val_div_sys_gnr: {round(div_sys_gnr, 5)}, val_div_usr_gnr: {round(div_usr_gnr, 5)}, val_div_cov_itm: {round(div_cov_itm, 5)}, val_div_cov_gnr: {round(div_cov_gnr, 5)}")
        train_losses.append(train_loss.item())
        val_losses.append(val_loss)
        val_precision.append(precision)
        val_recall.append(recall)
        val_ndcg.append(ndcg)
        val_srdp.append(srdp)
        val_nov .append(nov)
        val_div_sys_emb.append(div_sys_emb)
        val_div_usr_emb.append(div_usr_emb)
        val_div_sys_gnr.append(div_sys_gnr)
        val_div_usr_gnr.append(div_usr_gnr)
        val_div_cov_itm.append(div_cov_itm)
        val_div_cov_gnr.append(div_cov_gnr)
        model.train()

    if iter % ITERS_PER_LR_DECAY == 0 and iter != 0:
        scheduler.step()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(train_losses))]
plt.plot(iters, train_losses, label='train')
plt.plot(iters, val_losses, label='validation')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('training and validation loss curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_srdp))]
plt.plot(iters, val_srdp, label='srdp')
plt.plot(iters, val_recall, label='recall')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('srdp and recall curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_div_sys_emb))]
plt.plot(iters, val_div_sys_emb, label='div_sys_emb')
plt.plot(iters, val_div_usr_emb, label='div_usr_emb')
plt.plot(iters, val_div_sys_gnr, label='div_sys_gnr')
plt.ylabel('diversity score')
plt.title('diversity curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_div_sys_emb))]
plt.plot(iters, val_div_usr_gnr, label='div_usr_gnr')
plt.plot(iters, val_div_cov_itm, label='div_cov_itm')
plt.plot(iters, val_div_cov_gnr, label='div_cov_gnr')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('diversity curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_srdp))]
plt.plot(iters, val_ndcg, label='ndcg')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('ndcg curve')
plt.legend()
plt.show()

In [ ]:
# evaluate on test set
model.eval()
test_edge_index = test_edge_index.to(device)
test_sparse_edge_index = test_sparse_edge_index.to(device)

test_loss, test_recall, test_precision, test_ndcg, test_srdp, test_nov, test_div_sys_emb, test_div_usr_emb, test_div_sys_gnr, test_div_usr_gnr, test_div_cov_itm, test_div_cov_gnr= evaluation(
            model, test_edge_index, test_sparse_edge_index, [train_edge_index, val_edge_index], K, LAMBDA)

print(f"[test_loss: {round(test_loss, 5)}, test_recall@{K}: {round(test_recall, 5)}, test_precision@{K}: {round(test_precision, 5)}, test_ndcg@{K}: {round(test_ndcg, 5)}, test_srdp@{K}: {round(test_srdp, 5)}, test_nov@{K}: {round(test_nov, 5)}, test_div_sys_emb@{K}: {round(test_div_sys_emb, 5)}, test_div_usr_emb@{K}: {round(test_div_usr_emb, 5)}, test_div_sys_gnr@{K}: {round(test_div_sys_gnr, 5)}, test_div_usr_gnr@{K}: {round(test_div_usr_gnr, 5)}, test_div_cov_itm@{K}: {round(test_div_cov_itm, 5)}, test_div_cov_gnr@{K}: {round(test_div_cov_gnr, 5)}]")
print(f"{round(test_loss, 5)}, {round(test_recall, 5)},{round(test_precision, 5)},{round(test_ndcg, 5)},{round(test_srdp, 5)},{round(test_nov, 5)}, {round(test_div_sys_emb, 5)}, {round(test_div_usr_emb, 5)}, {round(test_div_sys_gnr, 5)}, {round(test_div_usr_gnr, 5)}, {round(test_div_cov_itm, 5)}, {round(test_div_cov_gnr, 5)}")

## Recommendations for a Given User

In [ ]:
#Load model
from pathlib import Path
data = 'ml-small'
text_model = 'roberta'
experiment = 'text'

# 1. Create models directory 
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# 2. Create model save path 
MODEL_NAME = "lightgcn_{0}_{1}.pth".format(experiment,text_model)

model = LightGCN(num_users, num_movies)
model.load_state_dict(torch.load(MODEL_PATH/data/text_model/MODEL_NAME)) # Set the model to inference mode

In [ ]:
model.eval()
df = pd.read_csv(movie_path)
movieid_title = pd.Series(df.Title.values,index=df.MovieID).to_dict()
movieid_genres = pd.Series(df.Genres.values,index=df.MovieID).to_dict()

user_pos_items = get_user_positive_items(edge_index)

In [ ]:
movieid_genres


In [ ]:
def make_predictions(user_id, num_recs):
    user = user_mapping[user_id]
    e_u = model.users_emb.weight[user]
    scores = model.items_emb.weight @ e_u

    values, indices = torch.topk(scores, k=len(user_pos_items[user]) + num_recs)

    movies = [index.cpu().item() for index in indices if index in user_pos_items[user]][:num_recs]
    movie_ids = [list(movie_mapping.keys())[list(movie_mapping.values()).index(movie)] for movie in movies]
    titles = [movieid_title[id] for id in movie_ids]
    genres = [movieid_genres[id] for id in movie_ids]

    print(f"Here are some movies that user {user_id} rated highly")
    for i in range(num_recs):
        print(f"title: {titles[i]}, genres: {genres[i]} ")

    print()

    movies = [index.cpu().item() for index in indices if index not in user_pos_items[user]][:num_recs]
    movie_ids = [list(movie_mapping.keys())[list(movie_mapping.values()).index(movie)] for movie in movies]
    titles = [movieid_title[id] for id in movie_ids]
    genres = [movieid_genres[id] for id in movie_ids]

    print(f"Here are some suggested movies for user {user_id}")
    for i in range(num_recs):
        print(f"title: {titles[i]}, genres: {genres[i]} ")

In [ ]:
USER_ID = 10
NUM_RECS = 10

make_predictions(USER_ID, NUM_RECS)

In [ ]:

from pathlib import Path
data = 'ml-small'
text_model = 'roberta'
experiment = 'text'

# 1. Create models directory 
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# 2. Create model save path 
MODEL_NAME = "lightgcn_{0}_{1}.pth".format(experiment,text_model)
MODEL_SAVE_PATH = MODEL_PATH / data / text_model / MODEL_NAME

# 3. Save the model state dict 
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model.state_dict(), # only saving the state_dict() only saves the models learned parameters
           f=MODEL_SAVE_PATH) 

# LightGCN with Image


## Fusion at Propagation

In [ ]:
# Integrating the embeddings at the initialization
import torch.nn.functional as F

class LightGCN_image(MessagePassing):

    def __init__(self, num_users, num_items, image_embeddings, embedding_dim=64, K=3, add_self_loops=False):
        super().__init__()
        self.num_users, self.num_items = num_users, num_items
        self.embedding_dim, self.K = embedding_dim, K
        self.add_self_loops = add_self_loops

        # Load precomputed text and image embeddings
        self.image_embeddings = image_embeddings  # Tensor of size [num_users + num_items, text_emb_dim]
        
        # Embedding layers for users and items
        self.users_emb = nn.Embedding(num_embeddings=self.num_users, embedding_dim=self.embedding_dim)
        self.items_emb = nn.Embedding(num_embeddings=self.num_items, embedding_dim=self.embedding_dim)

        # Initialize embeddings
        nn.init.normal_(self.users_emb.weight, std=0.1)
        nn.init.normal_(self.items_emb.weight, std=0.1)
        
        # Reducing dimensionality to balance
        self.dim_img_reduction = nn.Linear(self.image_embeddings.shape[1], self.embedding_dim)
       
        # Apply dimensionality reduction without making them trainable
        with torch.no_grad():
            reduced_image = self.dim_img_reduction(self.image_embeddings)
 
        self.image_embeddings = nn.Parameter(reduced_image, requires_grad=False)
 
    def forward(self, edge_index: SparseTensor):
        # Compute \tilde{A}: symmetrically normalized adjacency matrix
        edge_index_norm = gcn_norm(edge_index, add_self_loops=self.add_self_loops)

        # Trainable user and item embeddings
        emb_trainable_users = self.users_emb.weight
        emb_trainable_items = self.items_emb.weight
        
        # Concatenate trainable embeddings with fixed embeddings for users and items
        emb_users = torch.cat([emb_trainable_users, self.image_embeddings[:num_users]], dim=1)  # Users: trainable + fixed
        emb_items = torch.cat([emb_trainable_items,  self.image_embeddings[num_users:]], dim=1)  # Items: trainable + fixed

        # Initial embeddings, possibly reduced in dimension
        emb_0 = torch.cat([emb_users, emb_items])
        embs = [emb_0]
        emb_k = emb_0

        # Propagate embeddings
        for i in range(self.K):
            emb_k = self.propagate(edge_index_norm, x=emb_k)
            embs.append(emb_k)

        embs = torch.stack(embs, dim=1)
        emb_final = torch.mean(embs, dim=1)

        users_emb_final, items_emb_final = torch.split(emb_final, [self.num_users, self.num_items])

        return users_emb_final, self.users_emb.weight, items_emb_final, self.items_emb.weight
  

    def message(self, x_j: torch.Tensor) -> torch.Tensor:
        return x_j

    def message_and_aggregate(self, adj_t: SparseTensor, x: torch.Tensor) -> torch.Tensor:
        # Computes \tilde{A} @ x
        return matmul(adj_t, x)


## Fusion at Initialization

In [ ]:

# Integrating the embeddings at the initialization
import torch.nn.functional as F

class LightGCN_image(MessagePassing):

    def __init__(self, num_users, num_items, image_embeddings, embedding_dim=64, K=3, add_self_loops=False):
        super().__init__()
        self.num_users, self.num_items = num_users, num_items
        self.embedding_dim, self.K = embedding_dim, K
        self.add_self_loops = add_self_loops

        # Load precomputed text and image embeddings
        #self.text_embeddings = text_embeddings  # Tensor of size [num_users + num_items, text_emb_dim]
        self.image_embeddings = image_embeddings  # Tensor of size [num_users + num_items, image_emb_dim]

        # Embedding layers for users and items
        self.users_emb = nn.Embedding(num_embeddings=self.num_users, embedding_dim=self.embedding_dim)
        self.items_emb = nn.Embedding(num_embeddings=self.num_items, embedding_dim=self.embedding_dim)

        # Initialize embeddings
        nn.init.normal_(self.users_emb.weight, std=0.1)
        nn.init.normal_(self.items_emb.weight, std=0.1)
        
        # Reducing dimensionality to balance
        self.dim_img_reduction = nn.Linear(self.image_embeddings.shape[1], self.embedding_dim)
        self.image_embeddings = nn.Parameter(self.dim_img_reduction(self.image_embeddings))

        # Concatenate graph, text, and image embeddings
        self.users_emb.weight = nn.Parameter(torch.cat([self.users_emb.weight, self.image_embeddings[:self.num_users]], dim=1))
        self.items_emb.weight = nn.Parameter(torch.cat([self.items_emb.weight, self.image_embeddings[self.num_users:]], dim=1))
        
        # Reducing dimensionality of concatenated embeddings
        self.dim_usr_reduction = nn.Linear(self.users_emb.weight.shape[1], self.embedding_dim)
        self.users_emb.weight = nn.Parameter(self.dim_usr_reduction(self.users_emb.weight))
        self.dim_itm_reduction = nn.Linear(self.items_emb.weight.shape[1], self.embedding_dim)
        self.items_emb.weight = nn.Parameter(self.dim_itm_reduction(self.items_emb.weight))

    def forward(self, edge_index: SparseTensor):
        # Compute \tilde{A}: symmetrically normalized adjacency matrix
        edge_index_norm = gcn_norm(edge_index, add_self_loops=self.add_self_loops)

        # Initial embeddings, possibly reduced in dimension
        emb_0 = torch.cat([self.users_emb.weight, self.items_emb.weight])
        embs = [emb_0]
        emb_k = emb_0

        # Propagate embeddings
        for i in range(self.K):
            emb_k = self.propagate(edge_index_norm, x=emb_k)
            embs.append(emb_k)

        embs = torch.stack(embs, dim=1)
        emb_final = torch.mean(embs, dim=1)

        users_emb_final, items_emb_final = torch.split(emb_final, [self.num_users, self.num_items])

        return users_emb_final, self.users_emb.weight, items_emb_final, self.items_emb.weight
  

    def message(self, x_j: torch.Tensor) -> torch.Tensor:
        return x_j

    def message_and_aggregate(self, adj_t: SparseTensor, x: torch.Tensor) -> torch.Tensor:
        # Computes \tilde{A} @ x
        return matmul(adj_t, x)


## Training

In [ ]:
# Initialize the model
model = LightGCN_image(num_users=num_users, num_items=num_movies, image_embeddings=image_embeddings, embedding_dim=64)

In [ ]:
# define contants
ITERATIONS = 1000
BATCH_SIZE = 1024
LR = 1e-3
ITERS_PER_EVAL = 50
ITERS_PER_LR_DECAY = 50
K = 20
LAMBDA = 1e-6

In [ ]:
# setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device {device}.")


model = model.to(device)
model.train()

optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.95)

edge_index = edge_index.to(device)
train_edge_index = train_edge_index.to(device)
train_sparse_edge_index = train_sparse_edge_index.to(device)

val_edge_index = val_edge_index.to(device)
val_sparse_edge_index = val_sparse_edge_index.to(device)

In [ ]:
# training loop
train_losses = []
val_losses = []
val_precision = []
val_recall = []
val_ndcg = []
val_srdp = []
val_nov = []
val_div_sys_emb = []
val_div_usr_emb = []
val_div_sys_gnr = []
val_div_usr_gnr = []
val_div_cov_itm = []
val_div_cov_gnr = []

for iter in range(ITERATIONS):
 # forward propagation
    users_emb_final, users_emb_0, items_emb_final, items_emb_0 = model.forward(
        train_sparse_edge_index)

    # mini batching
    user_indices, pos_item_indices, neg_item_indices = sample_mini_batch(
        BATCH_SIZE, train_edge_index)
    user_indices, pos_item_indices, neg_item_indices = user_indices.to(
        device), pos_item_indices.to(device), neg_item_indices.to(device)
    users_emb_final, users_emb_0 = users_emb_final[user_indices], users_emb_0[user_indices]
    pos_items_emb_final, pos_items_emb_0 = items_emb_final[
        pos_item_indices], items_emb_0[pos_item_indices]
    neg_items_emb_final, neg_items_emb_0 = items_emb_final[
        neg_item_indices], items_emb_0[neg_item_indices]

    # loss computation
    train_loss = bpr_loss(users_emb_final, users_emb_0, pos_items_emb_final,
                          pos_items_emb_0, neg_items_emb_final, neg_items_emb_0, LAMBDA)

    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()

    if iter % ITERS_PER_EVAL == 0:
        model.eval()
        val_loss, recall, precision, ndcg, srdp, nov, div_sys_emb, div_usr_emb, div_sys_gnr, div_usr_gnr, div_cov_itm, div_cov_gnr = evaluation(
                model, val_edge_index, val_sparse_edge_index, [train_edge_index], K, LAMBDA)

        print(f"[Iteration {iter}/{ITERATIONS}] train_loss: {round(train_loss.item(), 5)}, val_loss: {round(val_loss, 5)}, val_recall@{K}: {round(recall, 5)}, val_precision@{K}: {round(precision, 5)}, val_ndcg@{K}: {round(ndcg, 5)}, val_srdp@{K}: {round(srdp, 5)}, val_nov@{K}: {round(nov, 5)}, val_div_sys_emb: {round(div_sys_emb, 5)}, val_div_usr_emb: {round(div_usr_emb, 5)}, val_div_sys_gnr: {round(div_sys_gnr, 5)}, val_div_usr_gnr: {round(div_usr_gnr, 5)}, val_div_cov_itm: {round(div_cov_itm, 5)}, val_div_cov_gnr: {round(div_cov_gnr, 5)}")
        train_losses.append(train_loss.item())
        val_losses.append(val_loss)
        val_precision.append(precision)
        val_recall.append(recall)
        val_ndcg.append(ndcg)
        val_srdp.append(srdp)
        val_nov .append(nov)
        val_div_sys_emb.append(div_sys_emb)
        val_div_usr_emb.append(div_usr_emb)
        val_div_sys_gnr.append(div_sys_gnr)
        val_div_usr_gnr.append(div_usr_gnr)
        val_div_cov_itm.append(div_cov_itm)
        val_div_cov_gnr.append(div_cov_gnr)
        model.train()

    if iter % ITERS_PER_LR_DECAY == 0 and iter != 0:
        scheduler.step()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(train_losses))]
plt.plot(iters, train_losses, label='train')
plt.plot(iters, val_losses, label='validation')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('training and validation loss curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_srdp))]
plt.plot(iters, val_srdp, label='srdp')
plt.plot(iters, val_recall, label='recall')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('srdp and recall curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_div_sys_emb))]
plt.plot(iters, val_div_sys_emb, label='div_sys_emb')
plt.plot(iters, val_div_usr_emb, label='div_usr_emb')
plt.plot(iters, val_div_sys_gnr, label='div_sys_gnr')
plt.ylabel('diversity score')
plt.title('diversity curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_div_sys_emb))]
plt.plot(iters, val_div_usr_gnr, label='div_usr_gnr')
plt.plot(iters, val_div_cov_itm, label='div_cov_itm')
plt.plot(iters, val_div_cov_gnr, label='div_cov_gnr')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('diversity curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_srdp))]
plt.plot(iters, val_ndcg, label='ndcg')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('ndcg curve')
plt.legend()
plt.show()

In [ ]:
# evaluate on test set
model.eval()
test_edge_index = test_edge_index.to(device)
test_sparse_edge_index = test_sparse_edge_index.to(device)

test_loss, test_recall, test_precision, test_ndcg, test_srdp, test_nov, test_div_sys_emb, test_div_usr_emb, test_div_sys_gnr, test_div_usr_gnr, test_div_cov_itm, test_div_cov_gnr= evaluation(
            model, test_edge_index, test_sparse_edge_index, [train_edge_index, val_edge_index], K, LAMBDA)

print(f"[test_loss: {round(test_loss, 5)}, test_recall@{K}: {round(test_recall, 5)}, test_precision@{K}: {round(test_precision, 5)}, test_ndcg@{K}: {round(test_ndcg, 5)}, test_srdp@{K}: {round(test_srdp, 5)}, test_nov@{K}: {round(test_nov, 5)}, test_div_sys_emb@{K}: {round(test_div_sys_emb, 5)}, test_div_usr_emb@{K}: {round(test_div_usr_emb, 5)}, test_div_sys_gnr@{K}: {round(test_div_sys_gnr, 5)}, test_div_usr_gnr@{K}: {round(test_div_usr_gnr, 5)}, test_div_cov_itm@{K}: {round(test_div_cov_itm, 5)}, test_div_cov_gnr@{K}: {round(test_div_cov_gnr, 5)}]")
print(f"{round(test_loss, 5)}, {round(test_recall, 5)},{round(test_precision, 5)},{round(test_ndcg, 5)},{round(test_srdp, 5)},{round(test_nov, 5)}, {round(test_div_sys_emb, 5)}, {round(test_div_usr_emb, 5)}, {round(test_div_sys_gnr, 5)}, {round(test_div_usr_gnr, 5)}, {round(test_div_cov_itm, 5)}, {round(test_div_cov_gnr, 5)}")

## Recommendations for a Given User

In [ ]:
#Load model
from pathlib import Path
data = 'ml-small'
text_model = 'roberta'
experiment = 'image'

# 1. Create models directory 
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# 2. Create model save path 
MODEL_NAME = "lightgcn_{0}_{1}.pth".format(experiment,text_model)

model = LightGCN(num_users, num_movies)
model.load_state_dict(torch.load(MODEL_PATH/data/text_model/MODEL_NAME)) # Set the model to inference mode

In [ ]:
model.eval()
df = pd.read_csv(movie_path)
movieid_title = pd.Series(df.Title.values,index=df.MovieID).to_dict()
movieid_genres = pd.Series(df.Genres.values,index=df.MovieID).to_dict()

user_pos_items = get_user_positive_items(edge_index)

In [ ]:
def make_predictions(user_id, num_recs):
    user = user_mapping[user_id]
    e_u = model.users_emb.weight[user]
    scores = model.items_emb.weight @ e_u

    values, indices = torch.topk(scores, k=len(user_pos_items[user]) + num_recs)

    movies = [index.cpu().item() for index in indices if index in user_pos_items[user]][:num_recs]
    movie_ids = [list(movie_mapping.keys())[list(movie_mapping.values()).index(movie)] for movie in movies]
    titles = [movieid_title[id] for id in movie_ids]
    genres = [movieid_genres[id] for id in movie_ids]

    print(f"Here are some movies that user {user_id} rated highly")
    for i in range(num_recs):
        print(f"title: {titles[i]}, genres: {genres[i]} ")

    print()

    movies = [index.cpu().item() for index in indices if index not in user_pos_items[user]][:num_recs]
    movie_ids = [list(movie_mapping.keys())[list(movie_mapping.values()).index(movie)] for movie in movies]
    titles = [movieid_title[id] for id in movie_ids]
    genres = [movieid_genres[id] for id in movie_ids]

    print(f"Here are some suggested movies for user {user_id}")
    for i in range(num_recs):
        print(f"title: {titles[i]}, genres: {genres[i]} ")

In [ ]:
USER_ID = 10
NUM_RECS = 10

make_predictions(USER_ID, NUM_RECS)

In [ ]:

from pathlib import Path
data = 'ml-small'
text_model = 'roberta'
experiment = 'image'

# 1. Create models directory 
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# 2. Create model save path 
MODEL_NAME = "lightgcn_{0}_{1}.pth".format(experiment,text_model)
MODEL_SAVE_PATH = MODEL_PATH / data / text_model / MODEL_NAME

# 3. Save the model state dict 
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model.state_dict(), # only saving the state_dict() only saves the models learned parameters
           f=MODEL_SAVE_PATH) 

# LightGCN with Text & Image

## Fusion at Propagation

In [ ]:
# Integrating the embeddings at the initialization
import torch.nn.functional as F

class LightGCN_combined(MessagePassing):

    def __init__(self, num_users, num_items, text_embeddings, image_embeddings, embedding_dim=64, K=3, add_self_loops=False):
        super().__init__()
        self.num_users, self.num_items = num_users, num_items
        self.embedding_dim, self.K = embedding_dim, K
        self.add_self_loops = add_self_loops

        # Load precomputed text and image embeddings
        self.text_embeddings = text_embeddings  # Tensor of size [num_users + num_items, text_emb_dim]
        self.image_embeddings = image_embeddings  # Tensor of size [num_users + num_items, image_emb_dim]

        # Embedding layers for users and items
        self.users_emb = nn.Embedding(num_embeddings=self.num_users, embedding_dim=self.embedding_dim)
        self.items_emb = nn.Embedding(num_embeddings=self.num_items, embedding_dim=self.embedding_dim)

        # Initialize embeddings
        nn.init.normal_(self.users_emb.weight, std=0.1)
        nn.init.normal_(self.items_emb.weight, std=0.1)
        
        # Reducing dimensionality to balance
        self.dim_txt_reduction = nn.Linear(self.text_embeddings.shape[1], self.embedding_dim)
        self.dim_img_reduction = nn.Linear(self.image_embeddings.shape[1], self.embedding_dim)
       
        # Apply dimensionality reduction without making them trainable
        with torch.no_grad():
            reduced_text = self.dim_txt_reduction(self.text_embeddings)
            reduced_image = self.dim_img_reduction(self.image_embeddings)

        self.text_embeddings = nn.Parameter(reduced_text, requires_grad=False)
        self.image_embeddings = nn.Parameter(reduced_image, requires_grad=False)

        # Concatenate text and image embeddings without making them trainable
        with torch.no_grad():
            self.concatenated_users = torch.cat([self.text_embeddings[:self.num_users], self.image_embeddings[:self.num_users]], dim=1)
            self.concatenated_items = torch.cat([self.text_embeddings[self.num_users:], self.image_embeddings[self.num_users:]], dim=1)

    def forward(self, edge_index: SparseTensor):
        # Compute \tilde{A}: symmetrically normalized adjacency matrix
        edge_index_norm = gcn_norm(edge_index, add_self_loops=self.add_self_loops)

        # Trainable user and item embeddings
        emb_trainable_users = self.users_emb.weight
        emb_trainable_items = self.items_emb.weight
        
        # Concatenate trainable embeddings with fixed embeddings for users and items
        emb_users = torch.cat([emb_trainable_users, self.concatenated_users], dim=1)  # Users: trainable + fixed
        emb_items = torch.cat([emb_trainable_items, self.concatenated_items], dim=1)  # Items: trainable + fixed

        # Initial embeddings, possibly reduced in dimension
        emb_0 = torch.cat([emb_users, emb_items])
        embs = [emb_0]
        emb_k = emb_0

        # Propagate embeddings
        for i in range(self.K):
            emb_k = self.propagate(edge_index_norm, x=emb_k)
            embs.append(emb_k)

        embs = torch.stack(embs, dim=1)
        emb_final = torch.mean(embs, dim=1)

        users_emb_final, items_emb_final = torch.split(emb_final, [self.num_users, self.num_items])

        return users_emb_final, self.users_emb.weight, items_emb_final, self.items_emb.weight
  

    def message(self, x_j: torch.Tensor) -> torch.Tensor:
        return x_j

    def message_and_aggregate(self, adj_t: SparseTensor, x: torch.Tensor) -> torch.Tensor:
        # Computes \tilde{A} @ x
        return matmul(adj_t, x)


## Fusion at Initialization

In [ ]:
# Integrating the embeddings at the initialization
import torch.nn.functional as F

class LightGCN_combined(MessagePassing):

    def __init__(self, num_users, num_items, text_embeddings, image_embeddings, embedding_dim=64, K=3, add_self_loops=False):
        super().__init__()
        self.num_users, self.num_items = num_users, num_items
        self.embedding_dim, self.K = embedding_dim, K
        self.add_self_loops = add_self_loops

        # Load precomputed text and image embeddings
        self.text_embeddings = text_embeddings  # Tensor of size [num_users + num_items, text_emb_dim]
        self.image_embeddings = image_embeddings  # Tensor of size [num_users + num_items, image_emb_dim]

        # Embedding layers for users and items
        self.users_emb = nn.Embedding(num_embeddings=self.num_users, embedding_dim=self.embedding_dim)
        self.items_emb = nn.Embedding(num_embeddings=self.num_items, embedding_dim=self.embedding_dim)

        # Initialize embeddings
        nn.init.normal_(self.users_emb.weight, std=0.1)
        nn.init.normal_(self.items_emb.weight, std=0.1)
        
        # Reducing dimensionality to balance
        self.dim_txt_reduction = nn.Linear(self.text_embeddings.shape[1], self.embedding_dim)
        self.dim_img_reduction = nn.Linear(self.image_embeddings.shape[1], self.embedding_dim)
        self.text_embeddings = nn.Parameter(self.dim_txt_reduction(self.text_embeddings))
        self.image_embeddings = nn.Parameter(self.dim_img_reduction(self.image_embeddings))

        # Concatenate graph, text, and image embeddings
        self.users_emb.weight = nn.Parameter(torch.cat([self.users_emb.weight, self.text_embeddings[:self.num_users], self.image_embeddings[:self.num_users]], dim=1))
        self.items_emb.weight = nn.Parameter(torch.cat([self.items_emb.weight, self.text_embeddings[self.num_users:], self.image_embeddings[self.num_users:]], dim=1))
        
        # Reducing dimensionality of concatenated embeddings
        #self.dim_usr_reduction = nn.Linear(self.users_emb.weight.shape[1], self.embedding_dim)
        #self.users_emb.weight = nn.Parameter(self.dim_usr_reduction(self.users_emb.weight))
        #self.dim_itm_reduction = nn.Linear(self.items_emb.weight.shape[1], self.embedding_dim)
        #self.items_emb.weight = nn.Parameter(self.dim_itm_reduction(self.items_emb.weight))


    def forward(self, edge_index: SparseTensor):
        # Compute \tilde{A}: symmetrically normalized adjacency matrix
        edge_index_norm = gcn_norm(edge_index, add_self_loops=self.add_self_loops)

        # Initial embeddings, possibly reduced in dimension
        emb_0 = torch.cat([self.users_emb.weight, self.items_emb.weight])
        embs = [emb_0]
        emb_k = emb_0

        # Propagate embeddings
        for i in range(self.K):
            emb_k = self.propagate(edge_index_norm, x=emb_k)
            embs.append(emb_k)

        embs = torch.stack(embs, dim=1)
        emb_final = torch.mean(embs, dim=1)

        users_emb_final, items_emb_final = torch.split(emb_final, [self.num_users, self.num_items])

        return users_emb_final, self.users_emb.weight, items_emb_final, self.items_emb.weight
  

    def message(self, x_j: torch.Tensor) -> torch.Tensor:
        return x_j

    def message_and_aggregate(self, adj_t: SparseTensor, x: torch.Tensor) -> torch.Tensor:
        # Computes \tilde{A} @ x
        return matmul(adj_t, x)


## Training

In [ ]:
# Initialize the model
model = LightGCN_combined(num_users=num_users, num_items=num_movies, text_embeddings=text_embeddings, image_embeddings=image_embeddings, embedding_dim=64)

In [ ]:
# define contants
ITERATIONS = 1000
BATCH_SIZE = 1024
LR = 1e-3
ITERS_PER_EVAL = 50
ITERS_PER_LR_DECAY = 50
K = 20
LAMBDA = 1e-6

In [ ]:
# setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device {device}.")


model = model.to(device)
model.train()

optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.95)

edge_index = edge_index.to(device)
train_edge_index = train_edge_index.to(device)
train_sparse_edge_index = train_sparse_edge_index.to(device)

val_edge_index = val_edge_index.to(device)
val_sparse_edge_index = val_sparse_edge_index.to(device)

In [ ]:
# training loop
train_losses = []
val_losses = []
val_precision = []
val_recall = []
val_ndcg = []
val_srdp = []
val_nov = []
val_div_sys_emb = []
val_div_usr_emb = []
val_div_sys_gnr = []
val_div_usr_gnr = []
val_div_cov_itm = []
val_div_cov_gnr = []

for iter in range(ITERATIONS):
 # forward propagation
    users_emb_final, users_emb_0, items_emb_final, items_emb_0 = model.forward(
        train_sparse_edge_index)

    # mini batching
    user_indices, pos_item_indices, neg_item_indices = sample_mini_batch(
        BATCH_SIZE, train_edge_index)
    user_indices, pos_item_indices, neg_item_indices = user_indices.to(
        device), pos_item_indices.to(device), neg_item_indices.to(device)
    users_emb_final, users_emb_0 = users_emb_final[user_indices], users_emb_0[user_indices]
    pos_items_emb_final, pos_items_emb_0 = items_emb_final[
        pos_item_indices], items_emb_0[pos_item_indices]
    neg_items_emb_final, neg_items_emb_0 = items_emb_final[
        neg_item_indices], items_emb_0[neg_item_indices]

    # loss computation
    train_loss = bpr_loss(users_emb_final, users_emb_0, pos_items_emb_final,
                          pos_items_emb_0, neg_items_emb_final, neg_items_emb_0, LAMBDA)

    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()

    if iter % ITERS_PER_EVAL == 0:
        model.eval()
        val_loss, recall, precision, ndcg, srdp, nov, div_sys_emb, div_usr_emb, div_sys_gnr, div_usr_gnr, div_cov_itm, div_cov_gnr = evaluation(
                model, val_edge_index, val_sparse_edge_index, [train_edge_index], K, LAMBDA)

        print(f"[Iteration {iter}/{ITERATIONS}] train_loss: {round(train_loss.item(), 5)}, val_loss: {round(val_loss, 5)}, val_recall@{K}: {round(recall, 5)}, val_precision@{K}: {round(precision, 5)}, val_ndcg@{K}: {round(ndcg, 5)}, val_srdp@{K}: {round(srdp, 5)}, val_nov@{K}: {round(nov, 5)}, val_div_sys_emb: {round(div_sys_emb, 5)}, val_div_usr_emb: {round(div_usr_emb, 5)}, val_div_sys_gnr: {round(div_sys_gnr, 5)}, val_div_usr_gnr: {round(div_usr_gnr, 5)}, val_div_cov_itm: {round(div_cov_itm, 5)}, val_div_cov_gnr: {round(div_cov_gnr, 5)}")
        train_losses.append(train_loss.item())
        val_losses.append(val_loss)
        val_precision.append(precision)
        val_recall.append(recall)
        val_ndcg.append(ndcg)
        val_srdp.append(srdp)
        val_nov .append(nov)
        val_div_sys_emb.append(div_sys_emb)
        val_div_usr_emb.append(div_usr_emb)
        val_div_sys_gnr.append(div_sys_gnr)
        val_div_usr_gnr.append(div_usr_gnr)
        val_div_cov_itm.append(div_cov_itm)
        val_div_cov_gnr.append(div_cov_gnr)
        model.train()

    if iter % ITERS_PER_LR_DECAY == 0 and iter != 0:
        scheduler.step()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(train_losses))]
plt.plot(iters, train_losses, label='train')
plt.plot(iters, val_losses, label='validation')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('training and validation loss curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_srdp))]
plt.plot(iters, val_srdp, label='srdp')
plt.plot(iters, val_recall, label='recall')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('srdp and recall curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_div_sys_emb))]
plt.plot(iters, val_div_sys_emb, label='div_sys_emb')
plt.plot(iters, val_div_usr_emb, label='div_usr_emb')
plt.plot(iters, val_div_sys_gnr, label='div_sys_gnr')
plt.ylabel('diversity score')
plt.title('diversity curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_div_sys_emb))]
plt.plot(iters, val_div_usr_gnr, label='div_usr_gnr')
plt.plot(iters, val_div_cov_itm, label='div_cov_itm')
plt.plot(iters, val_div_cov_gnr, label='div_cov_gnr')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('diversity curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_srdp))]
plt.plot(iters, val_ndcg, label='ndcg')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('ndcg curve')
plt.legend()
plt.show()

In [ ]:
# evaluate on test set
model.eval()
test_edge_index = test_edge_index.to(device)
test_sparse_edge_index = test_sparse_edge_index.to(device)

test_loss, test_recall, test_precision, test_ndcg, test_srdp, test_nov, test_div_sys_emb, test_div_usr_emb, test_div_sys_gnr, test_div_usr_gnr, test_div_cov_itm, test_div_cov_gnr= evaluation(
            model, test_edge_index, test_sparse_edge_index, [train_edge_index, val_edge_index], K, LAMBDA)

print(f"[test_loss: {round(test_loss, 5)}, test_recall@{K}: {round(test_recall, 5)}, test_precision@{K}: {round(test_precision, 5)}, test_ndcg@{K}: {round(test_ndcg, 5)}, test_srdp@{K}: {round(test_srdp, 5)}, test_nov@{K}: {round(test_nov, 5)}, test_div_sys_emb@{K}: {round(test_div_sys_emb, 5)}, test_div_usr_emb@{K}: {round(test_div_usr_emb, 5)}, test_div_sys_gnr@{K}: {round(test_div_sys_gnr, 5)}, test_div_usr_gnr@{K}: {round(test_div_usr_gnr, 5)}, test_div_cov_itm@{K}: {round(test_div_cov_itm, 5)}, test_div_cov_gnr@{K}: {round(test_div_cov_gnr, 5)}]")
print(f"{round(test_loss, 5)}, {round(test_recall, 5)},{round(test_precision, 5)},{round(test_ndcg, 5)},{round(test_srdp, 5)},{round(test_nov, 5)}, {round(test_div_sys_emb, 5)}, {round(test_div_usr_emb, 5)}, {round(test_div_sys_gnr, 5)}, {round(test_div_usr_gnr, 5)}, {round(test_div_cov_itm, 5)}, {round(test_div_cov_gnr, 5)}")

## Recommendations for a Given User

In [ ]:
#Load model
from pathlib import Path
data = 'ml-small'
text_model = 'roberta'
experiment = 'text_image'

# 1. Create models directory 
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# 2. Create model save path 
MODEL_NAME = "lightgcn_{0}_{1}.pth".format(experiment,text_model)

model = LightGCN(num_users, num_movies)
model.load_state_dict(torch.load(MODEL_PATH/data/text_model/MODEL_NAME)) # Set the model to inference mode

In [ ]:
model.eval()
df = pd.read_csv(movie_path)
movieid_title = pd.Series(df.Title.values,index=df.MovieID).to_dict()
movieid_genres = pd.Series(df.Genres.values,index=df.MovieID).to_dict()

user_pos_items = get_user_positive_items(edge_index)

In [ ]:
def make_predictions(user_id, num_recs):
    user = user_mapping[user_id]
    e_u = model.users_emb.weight[user]
    scores = model.items_emb.weight @ e_u

    values, indices = torch.topk(scores, k=len(user_pos_items[user]) + num_recs)

    movies = [index.cpu().item() for index in indices if index in user_pos_items[user]][:num_recs]
    movie_ids = [list(movie_mapping.keys())[list(movie_mapping.values()).index(movie)] for movie in movies]
    titles = [movieid_title[id] for id in movie_ids]
    genres = [movieid_genres[id] for id in movie_ids]

    print(f"Here are some movies that user {user_id} rated highly")
    for i in range(num_recs):
        print(f"title: {titles[i]}, genres: {genres[i]} ")

    print()

    movies = [index.cpu().item() for index in indices if index not in user_pos_items[user]][:num_recs]
    movie_ids = [list(movie_mapping.keys())[list(movie_mapping.values()).index(movie)] for movie in movies]
    titles = [movieid_title[id] for id in movie_ids]
    genres = [movieid_genres[id] for id in movie_ids]

    print(f"Here are some suggested movies for user {user_id}")
    for i in range(num_recs):
        print(f"title: {titles[i]}, genres: {genres[i]} ")

In [ ]:
USER_ID = 10
NUM_RECS = 10

make_predictions(USER_ID, NUM_RECS)

In [ ]:

from pathlib import Path
data = 'ml-small'
text_model = 'roberta'
experiment = 'text_image'

# 1. Create models directory 
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# 2. Create model save path 
MODEL_NAME = "lightgcn_{0}_{1}.pth".format(experiment,text_model)
MODEL_SAVE_PATH = MODEL_PATH / data / text_model / MODEL_NAME

# 3. Save the model state dict 
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model.state_dict(), # only saving the state_dict() only saves the models learned parameters
           f=MODEL_SAVE_PATH) 

# LightGCN no ID, with Text

## Fusion at Propagation

In [ ]:
# Integrating the embeddings at the forward pass
import torch.nn.functional as F

class LightGCN_combined_noID(MessagePassing):

    def __init__(self, num_users, num_items, text_embeddings, image_embeddings, embedding_dim=64, K=3, add_self_loops=False):
        super().__init__()
        self.num_users, self.num_items = num_users, num_items
        self.embedding_dim, self.K = embedding_dim, K
        self.add_self_loops = add_self_loops
        
        # Load precomputed text and image embeddings
        self.text_embeddings = text_embeddings
        self.image_embeddings = image_embeddings

        # Embedding layers for users and items
        
        self.users_emb = nn.Embedding(num_users, embedding_dim)
        self.items_emb = nn.Embedding(num_items, embedding_dim)
        
        #self.users_emb = self.image_embeddings[:self.num_users]
        #self.items_emb = self.image_embeddings[self.num_users:]
        
        # Initialize embeddings
        #nn.init.normal_(self.users_emb.weight, std=0.1)
        #nn.init.normal_(self.items_emb.weight, std=0.1)
        
        # Reducing dimensionality to balance
        #self.dim_txt_reduction = nn.Linear(self.text_embeddings.shape[1], self.embedding_dim)
        #self.dim_img_reduction = nn.Linear(self.image_embeddings.shape[1], self.embedding_dim)
       
        # Apply dimensionality reduction without making them trainable
        #with torch.no_grad():
        #    reduced_text = self.dim_txt_reduction(self.text_embeddings)
        #    reduced_image = self.dim_img_reduction(self.image_embeddings)

        #self.text_embeddings = nn.Parameter(reduced_text, requires_grad=False)
        #self.image_embeddings = nn.Parameter(reduced_image, requires_grad=False)

        # Concatenate text and image embeddings without making them trainable
        #with torch.no_grad():
        #    self.concatenated_users = torch.cat([self.text_embeddings[:self.num_users], self.image_embeddings[:self.num_users]], dim=1)
        #    self.concatenated_items = torch.cat([self.text_embeddings[self.num_users:], self.image_embeddings[self.num_users:]], dim=1)

    def forward(self, edge_index: SparseTensor):
        # Compute \tilde{A}: symmetrically normalized adjacency matrix
        edge_index_norm = gcn_norm(edge_index, add_self_loops=self.add_self_loops)

        # Trainable user and item embeddings
        # emb_trainable_users = self.users_emb.weight
        # emb_trainable_items = self.items_emb.weight
        
        # Concatenate trainable embeddings with fixed embeddings for users and items
        with torch.no_grad():
            emb_users = torch.cat([self.image_embeddings[:self.num_users],self.text_embeddings[:self.num_users]], dim=1,)  # Users: fixed
            emb_items = torch.cat([self.image_embeddings[self.num_users:],self.text_embeddings[self.num_users:]], dim=1)  # Items: fixed

        # Initial embeddings, possibly reduced in dimension
        emb_0 = torch.cat([emb_users, emb_items])
        embs = [emb_0]
        emb_k = emb_0

        # Propagate embeddings
        for i in range(self.K):
            emb_k = self.propagate(edge_index_norm, x=emb_k)
            embs.append(emb_k)

        embs = torch.stack(embs, dim=1)
        emb_final = torch.mean(embs, dim=1)

        users_emb_final, items_emb_final = torch.split(emb_final, [self.num_users, self.num_items])

        return users_emb_final, self.users_emb.weight, items_emb_final, self.items_emb.weight

    def message(self, x_j: torch.Tensor) -> torch.Tensor:
        return x_j

    def message_and_aggregate(self, adj_t: SparseTensor, x: torch.Tensor) -> torch.Tensor:
        # Computes \tilde{A} @ x
        return matmul(adj_t, x)


## Fusion at Initialization

In [ ]:
# Integrating the embeddings at the initialization
import torch.nn.functional as F

class LightGCN_combined_noID(MessagePassing):

    def __init__(self, num_users, num_items, text_embeddings, image_embeddings, embedding_dim=64, K=3, add_self_loops=False):
        super().__init__()
        self.num_users, self.num_items = num_users, num_items
        self.embedding_dim, self.K = embedding_dim, K
        self.add_self_loops = add_self_loops

        # Load precomputed text and image embeddings
        self.text_embeddings = text_embeddings  # Tensor of size [num_users + num_items, text_emb_dim]
        self.image_embeddings = image_embeddings  # Tensor of size [num_users + num_items, image_emb_dim]

        # Embedding layers for users and items
        self.users_emb = nn.Embedding(num_embeddings=self.num_users, embedding_dim=self.embedding_dim)
        self.items_emb = nn.Embedding(num_embeddings=self.num_items, embedding_dim=self.embedding_dim)

        # Initialize embeddings
        nn.init.normal_(self.users_emb.weight, std=0.1)
        nn.init.normal_(self.items_emb.weight, std=0.1)
        
        # Reducing dimensionality to balance
        self.dim_txt_reduction = nn.Linear(self.text_embeddings.shape[1], self.embedding_dim)
        self.dim_img_reduction = nn.Linear(self.image_embeddings.shape[1], self.embedding_dim)
        #self.text_embeddings = nn.Parameter(self.dim_txt_reduction(self.text_embeddings))
        #self.image_embeddings = nn.Parameter(self.dim_img_reduction(self.image_embeddings))

        # Apply dimensionality reduction without making them trainable
        with torch.no_grad():
            reduced_text = self.dim_txt_reduction(self.text_embeddings)
            reduced_image = self.dim_img_reduction(self.image_embeddings)

        self.text_embeddings = nn.Parameter(reduced_text, requires_grad=False)
        self.image_embeddings = nn.Parameter(reduced_image, requires_grad=False)

        # Concatenate text and image embeddings without making them trainable
        with torch.no_grad():
            concatenated_users = torch.cat([self.text_embeddings[:self.num_users], self.image_embeddings[:self.num_users]], dim=1)
            concatenated_items = torch.cat([self.text_embeddings[self.num_users:], self.image_embeddings[self.num_users:]], dim=1)

        self.users_embeddings = nn.Parameter(concatenated_users, requires_grad=False)
        self.items_embeddings = nn.Parameter(concatenated_items, requires_grad=False)

    def forward(self, edge_index: SparseTensor):
        # Compute \tilde{A}: symmetrically normalized adjacency matrix
        edge_index_norm = gcn_norm(edge_index, add_self_loops=self.add_self_loops)

        # Initial embeddings, possibly reduced in dimension
        emb_0 = torch.cat([self.users_emb.weight, self.items_emb.weight])
        embs = [emb_0]
        emb_k = emb_0

        # Propagate embeddings
        for i in range(self.K):
            emb_k = self.propagate(edge_index_norm, x=emb_k)
            embs.append(emb_k)

        embs = torch.stack(embs, dim=1)
        emb_final = torch.mean(embs, dim=1)

        users_emb_final, items_emb_final = torch.split(emb_final, [self.num_users, self.num_items])

        return users_emb_final, self.users_emb.weight, items_emb_final, self.items_emb.weight
  

    def message(self, x_j: torch.Tensor) -> torch.Tensor:
        return x_j

    def message_and_aggregate(self, adj_t: SparseTensor, x: torch.Tensor) -> torch.Tensor:
        # Computes \tilde{A} @ x
        return matmul(adj_t, x)


## Training

In [ ]:
# Initialize the model
model = LightGCN_combined_noID(num_users=num_users, num_items=num_movies, text_embeddings=text_embeddings, image_embeddings=image_embeddings, embedding_dim=64)

In [ ]:
# define contants
ITERATIONS = 1000
BATCH_SIZE = 1024
LR = 1e-3
ITERS_PER_EVAL = 50
ITERS_PER_LR_DECAY = 50
K = 20
LAMBDA = 1e-6

In [ ]:
# setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device {device}.")


model = model.to(device)
model.train()

optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.95)

edge_index = edge_index.to(device)
train_edge_index = train_edge_index.to(device)
train_sparse_edge_index = train_sparse_edge_index.to(device)

val_edge_index = val_edge_index.to(device)
val_sparse_edge_index = val_sparse_edge_index.to(device)

In [ ]:
# training loop
train_losses = []
val_losses = []
val_precision = []
val_recall = []
val_ndcg = []
val_srdp = []
val_nov = []


for iter in range(ITERATIONS):
 # forward propagation
    users_emb_final, users_emb_0, items_emb_final, items_emb_0 = model.forward(
        train_sparse_edge_index)

    # mini batching
    user_indices, pos_item_indices, neg_item_indices = sample_mini_batch(
        BATCH_SIZE, train_edge_index)
    user_indices, pos_item_indices, neg_item_indices = user_indices.to(
        device), pos_item_indices.to(device), neg_item_indices.to(device)
    users_emb_final, users_emb_0 = users_emb_final[user_indices], users_emb_0[user_indices]
    pos_items_emb_final, pos_items_emb_0 = items_emb_final[
        pos_item_indices], items_emb_0[pos_item_indices]
    neg_items_emb_final, neg_items_emb_0 = items_emb_final[
        neg_item_indices], items_emb_0[neg_item_indices]

    # loss computation
    train_loss = bpr_loss(users_emb_final, users_emb_0, pos_items_emb_final,
                          pos_items_emb_0, neg_items_emb_final, neg_items_emb_0, LAMBDA)

    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()

    if iter % ITERS_PER_EVAL == 0:
        model.eval()
        val_loss, recall, precision, ndcg, srdp, nov = evaluation(
                model, val_edge_index, val_sparse_edge_index, [train_edge_index], K, LAMBDA)

        print(f"[Iteration {iter}/{ITERATIONS}] train_loss: {round(train_loss.item(), 5)}, val_loss: {round(val_loss, 5)}, val_recall@{K}: {round(recall, 5)}, val_precision@{K}: {round(precision, 5)}, val_ndcg@{K}: {round(ndcg, 5)}, val_srdp@{K}: {round(srdp, 5)}, val_nov@{K}: {round(nov, 5)}")
        train_losses.append(train_loss.item())
        val_losses.append(val_loss)
        val_precision.append(precision)
        val_recall.append(recall)
        val_ndcg.append(ndcg)
        val_srdp.append(srdp)
        val_nov .append(nov)
        model.train()

    if iter % ITERS_PER_LR_DECAY == 0 and iter != 0:
        scheduler.step()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(train_losses))]
plt.plot(iters, train_losses, label='train')
plt.plot(iters, val_losses, label='validation')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('training and validation loss curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_srdp))]
plt.plot(iters, val_srdp, label='srdp')
plt.plot(iters, val_recall, label='recall')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('srdp and recall curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_srdp))]
plt.plot(iters, val_ndcg, label='ndcg')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('ndcg curve')
plt.legend()
plt.show()

In [ ]:
# evaluate on test set
model.eval()
test_edge_index = test_edge_index.to(device)
test_sparse_edge_index = test_sparse_edge_index.to(device)

test_loss, test_recall, test_precision, test_ndcg, test_srdp, test_nov = evaluation(
            model, test_edge_index, test_sparse_edge_index, [train_edge_index, val_edge_index], K, LAMBDA)

print(f"[test_loss: {round(test_loss, 5)}, test_recall@{K}: {round(test_recall, 5)}, test_precision@{K}: {round(test_precision, 5)}, test_ndcg@{K}: {round(test_ndcg, 5)}, test_srdp@{K}: {round(test_srdp, 5)}, test_nov@{K}: {round(test_nov, 5)}")
print(f"{round(test_loss, 5)}, {round(test_recall, 5)},{round(test_precision, 5)},{round(test_ndcg, 5)},{round(test_srdp, 5)},{round(test_nov, 5)}")

# LightGCN no ID, with Text

## Fusion at Propagation

In [ ]:
# Integrating the embeddings at the forward pass
import torch.nn.functional as F

class LightGCN_text_noID(MessagePassing):

    def __init__(self, num_users, num_items, text_embeddings, image_embeddings, embedding_dim=64, K=3, add_self_loops=False):
        super().__init__()
        self.num_users, self.num_items = num_users, num_items
        self.embedding_dim, self.K = embedding_dim, K
        self.add_self_loops = add_self_loops
        
        # Load precomputed text and image embeddings
        self.text_embeddings = text_embeddings
        self.image_embeddings = image_embeddings

        # Embedding layers for users and items
        
        self.users_emb = nn.Embedding(num_users, embedding_dim)
        self.items_emb = nn.Embedding(num_items, embedding_dim)
        
        #self.users_emb = self.image_embeddings[:self.num_users]
        #self.items_emb = self.image_embeddings[self.num_users:]
        
        # Initialize embeddings
        #nn.init.normal_(self.users_emb.weight, std=0.1)
        #nn.init.normal_(self.items_emb.weight, std=0.1)
        
        # Reducing dimensionality to balance
        #self.dim_txt_reduction = nn.Linear(self.text_embeddings.shape[1], self.embedding_dim)
        #self.dim_img_reduction = nn.Linear(self.image_embeddings.shape[1], self.embedding_dim)
       
        # Apply dimensionality reduction without making them trainable
        #with torch.no_grad():
        #    reduced_text = self.dim_txt_reduction(self.text_embeddings)
        #    reduced_image = self.dim_img_reduction(self.image_embeddings)

        #self.text_embeddings = nn.Parameter(reduced_text, requires_grad=False)
        #self.image_embeddings = nn.Parameter(reduced_image, requires_grad=False)

        # Concatenate text and image embeddings without making them trainable
        #with torch.no_grad():
        #    self.concatenated_users = torch.cat([self.text_embeddings[:self.num_users], self.image_embeddings[:self.num_users]], dim=1)
        #    self.concatenated_items = torch.cat([self.text_embeddings[self.num_users:], self.image_embeddings[self.num_users:]], dim=1)

    def forward(self, edge_index: SparseTensor):
        # Compute \tilde{A}: symmetrically normalized adjacency matrix
        edge_index_norm = gcn_norm(edge_index, add_self_loops=self.add_self_loops)

        # Trainable user and item embeddings
        # emb_trainable_users = self.users_emb.weight
        # emb_trainable_items = self.items_emb.weight
        
        # Concatenate trainable embeddings with fixed embeddings for users and items
        #emb_users = self.image_embeddings[:self.num_users]  # Users: fixed
        #emb_items = self.image_embeddings[self.num_users:]  # Items: fixed

        # Initial embeddings, possibly reduced in dimension
        emb_0 = torch.cat([self.text_embeddings[:self.num_users], self.text_embeddings[self.num_users:]])
        embs = [emb_0]
        emb_k = emb_0

        # Propagate embeddings
        for i in range(self.K):
            emb_k = self.propagate(edge_index_norm, x=emb_k)
            embs.append(emb_k)

        embs = torch.stack(embs, dim=1)
        emb_final = torch.mean(embs, dim=1)

        users_emb_final, items_emb_final = torch.split(emb_final, [self.num_users, self.num_items])

        return users_emb_final, self.users_emb.weight, items_emb_final, self.items_emb.weight

    def message(self, x_j: torch.Tensor) -> torch.Tensor:
        return x_j

    def message_and_aggregate(self, adj_t: SparseTensor, x: torch.Tensor) -> torch.Tensor:
        # Computes \tilde{A} @ x
        return matmul(adj_t, x)


## Fusion at Initialization

In [ ]:
# Integrating the embeddings at the initialization
import torch.nn.functional as F

class LightGCN_text_noID(MessagePassing):

    def __init__(self, num_users, num_items, text_embeddings, image_embeddings, embedding_dim=64, K=3, add_self_loops=False):
        super().__init__()
        self.num_users, self.num_items = num_users, num_items
        self.embedding_dim, self.K = embedding_dim, K
        self.add_self_loops = add_self_loops

        # Load precomputed text and image embeddings
        self.text_embeddings = text_embeddings  # Tensor of size [num_users + num_items, text_emb_dim]
        self.image_embeddings = image_embeddings  # Tensor of size [num_users + num_items, image_emb_dim]

        # Embedding layers for users and items
        self.users_emb = nn.Embedding(num_embeddings=self.num_users, embedding_dim=self.embedding_dim)
        self.items_emb = nn.Embedding(num_embeddings=self.num_items, embedding_dim=self.embedding_dim)

        # Initialize embeddings
        nn.init.normal_(self.users_emb.weight, std=0.1)
        nn.init.normal_(self.items_emb.weight, std=0.1)
        
        # Reducing dimensionality to balance
        self.dim_txt_reduction = nn.Linear(self.text_embeddings.shape[1], self.embedding_dim)
        self.dim_img_reduction = nn.Linear(self.image_embeddings.shape[1], self.embedding_dim)
        #self.text_embeddings = nn.Parameter(self.dim_txt_reduction(self.text_embeddings))
        #self.image_embeddings = nn.Parameter(self.dim_img_reduction(self.image_embeddings))

       # Apply dimensionality reduction without making them trainable
        with torch.no_grad():
            reduced_text = self.dim_txt_reduction(self.text_embeddings)
            #reduced_image = self.dim_img_reduction(self.image_embeddings)

        self.text_embeddings = nn.Parameter(reduced_text, requires_grad=False)
        #self.image_embeddings = nn.Parameter(reduced_image, requires_grad=False)

        # Concatenate text and image embeddings without making them trainable
        self.users_embeddings = nn.Parameter(self.text_embeddings[:self.num_users], requires_grad=False)
        self.items_embeddings = nn.Parameter(self.text_embeddings[self.num_users:], requires_grad=False)
    

    def forward(self, edge_index: SparseTensor):
        # Compute \tilde{A}: symmetrically normalized adjacency matrix
        edge_index_norm = gcn_norm(edge_index, add_self_loops=self.add_self_loops)

        # Initial embeddings, possibly reduced in dimension
        emb_0 = torch.cat([self.users_emb.weight, self.items_emb.weight])
        embs = [emb_0]
        emb_k = emb_0

        # Propagate embeddings
        for i in range(self.K):
            emb_k = self.propagate(edge_index_norm, x=emb_k)
            embs.append(emb_k)

        embs = torch.stack(embs, dim=1)
        emb_final = torch.mean(embs, dim=1)

        users_emb_final, items_emb_final = torch.split(emb_final, [self.num_users, self.num_items])

        return users_emb_final, self.users_emb.weight, items_emb_final, self.items_emb.weight
  

    def message(self, x_j: torch.Tensor) -> torch.Tensor:
        return x_j

    def message_and_aggregate(self, adj_t: SparseTensor, x: torch.Tensor) -> torch.Tensor:
        # Computes \tilde{A} @ x
        return matmul(adj_t, x)


## Training

In [ ]:
# Initialize the model
model = LightGCN_text_noID(num_users=num_users, num_items=num_movies, text_embeddings=text_embeddings, image_embeddings=image_embeddings, embedding_dim=64)

In [ ]:
# define contants
ITERATIONS = 1000
BATCH_SIZE = 1024
LR = 1e-3
ITERS_PER_EVAL = 50
ITERS_PER_LR_DECAY = 50
K = 20
LAMBDA = 1e-6

In [ ]:
# setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device {device}.")


model = model.to(device)
model.train()

optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.95)

edge_index = edge_index.to(device)
train_edge_index = train_edge_index.to(device)
train_sparse_edge_index = train_sparse_edge_index.to(device)

val_edge_index = val_edge_index.to(device)
val_sparse_edge_index = val_sparse_edge_index.to(device)

In [ ]:
# training loop
train_losses = []
val_losses = []
val_precision = []
val_recall = []
val_ndcg = []
val_srdp = []
val_nov = []


for iter in range(ITERATIONS):
 # forward propagation
    users_emb_final, users_emb_0, items_emb_final, items_emb_0 = model.forward(
        train_sparse_edge_index)

    # mini batching
    user_indices, pos_item_indices, neg_item_indices = sample_mini_batch(
        BATCH_SIZE, train_edge_index)
    user_indices, pos_item_indices, neg_item_indices = user_indices.to(
        device), pos_item_indices.to(device), neg_item_indices.to(device)
    users_emb_final, users_emb_0 = users_emb_final[user_indices], users_emb_0[user_indices]
    pos_items_emb_final, pos_items_emb_0 = items_emb_final[
        pos_item_indices], items_emb_0[pos_item_indices]
    neg_items_emb_final, neg_items_emb_0 = items_emb_final[
        neg_item_indices], items_emb_0[neg_item_indices]

    # loss computation
    train_loss = bpr_loss(users_emb_final, users_emb_0, pos_items_emb_final,
                          pos_items_emb_0, neg_items_emb_final, neg_items_emb_0, LAMBDA)

    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()

    if iter % ITERS_PER_EVAL == 0:
        model.eval()
        val_loss, recall, precision, ndcg, srdp, nov = evaluation(
                model, val_edge_index, val_sparse_edge_index, [train_edge_index], K, LAMBDA)

        print(f"[Iteration {iter}/{ITERATIONS}] train_loss: {round(train_loss.item(), 5)}, val_loss: {round(val_loss, 5)}, val_recall@{K}: {round(recall, 5)}, val_precision@{K}: {round(precision, 5)}, val_ndcg@{K}: {round(ndcg, 5)}, val_srdp@{K}: {round(srdp, 5)}, val_nov@{K}: {round(nov, 5)}")
        train_losses.append(train_loss.item())
        val_losses.append(val_loss)
        val_precision.append(precision)
        val_recall.append(recall)
        val_ndcg.append(ndcg)
        val_srdp.append(srdp)
        val_nov .append(nov)
        model.train()

    if iter % ITERS_PER_LR_DECAY == 0 and iter != 0:
        scheduler.step()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(train_losses))]
plt.plot(iters, train_losses, label='train')
plt.plot(iters, val_losses, label='validation')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('training and validation loss curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_srdp))]
plt.plot(iters, val_srdp, label='srdp')
plt.plot(iters, val_recall, label='recall')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('srdp and recall curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_srdp))]
plt.plot(iters, val_ndcg, label='ndcg')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('ndcg curve')
plt.legend()
plt.show()

In [ ]:
# evaluate on test set
model.eval()
test_edge_index = test_edge_index.to(device)
test_sparse_edge_index = test_sparse_edge_index.to(device)

test_loss, test_recall, test_precision, test_ndcg, test_srdp, test_nov = evaluation(
            model, test_edge_index, test_sparse_edge_index, [train_edge_index, val_edge_index], K, LAMBDA)

print(f"[test_loss: {round(test_loss, 5)}, test_recall@{K}: {round(test_recall, 5)}, test_precision@{K}: {round(test_precision, 5)}, test_ndcg@{K}: {round(test_ndcg, 5)}, test_srdp@{K}: {round(test_srdp, 5)}, test_nov@{K}: {round(test_nov, 5)}")
print(f"{round(test_loss, 5)}, {round(test_recall, 5)},{round(test_precision, 5)},{round(test_ndcg, 5)},{round(test_srdp, 5)},{round(test_nov, 5)}")

# LightGCN no ID, with Image

## Fusion at Propagation

In [ ]:
# Integrating the embeddings at the forward pass
import torch.nn.functional as F

class LightGCN_image_noID(MessagePassing):

    def __init__(self, num_users, num_items, text_embeddings, image_embeddings, embedding_dim=64, K=3, add_self_loops=False):
        super().__init__()
        self.num_users, self.num_items = num_users, num_items
        self.embedding_dim, self.K = embedding_dim, K
        self.add_self_loops = add_self_loops
        
        # Load precomputed text and image embeddings
        self.text_embeddings = text_embeddings
        self.image_embeddings = image_embeddings

        # Embedding layers for users and items
        
        self.users_emb = nn.Embedding(num_users, embedding_dim)
        self.items_emb = nn.Embedding(num_items, embedding_dim)
        
        #self.users_emb = self.image_embeddings[:self.num_users]
        #self.items_emb = self.image_embeddings[self.num_users:]
        
        # Initialize embeddings
        #nn.init.normal_(self.users_emb.weight, std=0.1)
        #nn.init.normal_(self.items_emb.weight, std=0.1)
        
        # Reducing dimensionality to balance
        #self.dim_txt_reduction = nn.Linear(self.text_embeddings.shape[1], self.embedding_dim)
        #self.dim_img_reduction = nn.Linear(self.image_embeddings.shape[1], self.embedding_dim)
       
        # Apply dimensionality reduction without making them trainable
        #with torch.no_grad():
        #    reduced_text = self.dim_txt_reduction(self.text_embeddings)
        #    reduced_image = self.dim_img_reduction(self.image_embeddings)

        #self.text_embeddings = nn.Parameter(reduced_text, requires_grad=False)
        #self.image_embeddings = nn.Parameter(reduced_image, requires_grad=False)

        # Concatenate text and image embeddings without making them trainable
        #with torch.no_grad():
        #    self.concatenated_users = torch.cat([self.text_embeddings[:self.num_users], self.image_embeddings[:self.num_users]], dim=1)
        #    self.concatenated_items = torch.cat([self.text_embeddings[self.num_users:], self.image_embeddings[self.num_users:]], dim=1)

    def forward(self, edge_index: SparseTensor):
        # Compute \tilde{A}: symmetrically normalized adjacency matrix
        edge_index_norm = gcn_norm(edge_index, add_self_loops=self.add_self_loops)

        # Trainable user and item embeddings
        # emb_trainable_users = self.users_emb.weight
        # emb_trainable_items = self.items_emb.weight
        
        # Concatenate trainable embeddings with fixed embeddings for users and items
        #emb_users = self.image_embeddings[:self.num_users]  # Users: fixed
        #emb_items = self.image_embeddings[self.num_users:]  # Items: fixed

        # Initial embeddings, possibly reduced in dimension
        emb_0 = torch.cat([self.image_embeddings[:self.num_users], self.image_embeddings[self.num_users:]])
        embs = [emb_0]
        emb_k = emb_0

        # Propagate embeddings
        for i in range(self.K):
            emb_k = self.propagate(edge_index_norm, x=emb_k)
            embs.append(emb_k)

        embs = torch.stack(embs, dim=1)
        emb_final = torch.mean(embs, dim=1)

        users_emb_final, items_emb_final = torch.split(emb_final, [self.num_users, self.num_items])

        return users_emb_final, self.users_emb.weight, items_emb_final, self.items_emb.weight

    def message(self, x_j: torch.Tensor) -> torch.Tensor:
        return x_j

    def message_and_aggregate(self, adj_t: SparseTensor, x: torch.Tensor) -> torch.Tensor:
        # Computes \tilde{A} @ x
        return matmul(adj_t, x)


In [ ]:
text_embeddings[1].shape[0]

## Fusion at Initialization

In [ ]:
# Integrating the embeddings at the initialization
import torch.nn.functional as F

class LightGCN_image_noID(MessagePassing):

    def __init__(self, num_users, num_items, text_embeddings, image_embeddings, embedding_dim=64, K=3, add_self_loops=False):
        super().__init__()
        self.num_users, self.num_items = num_users, num_items
        self.embedding_dim, self.K = embedding_dim, K
        self.add_self_loops = add_self_loops

        # Load precomputed text and image embeddings
        self.text_embeddings = text_embeddings  # Tensor of size [num_users + num_items, text_emb_dim]
        self.image_embeddings = image_embeddings  # Tensor of size [num_users + num_items, image_emb_dim]

        # Embedding layers for users and items
        self.users_emb = nn.Embedding(num_embeddings=self.num_users, embedding_dim=self.embedding_dim)
        self.items_emb = nn.Embedding(num_embeddings=self.num_items, embedding_dim=self.embedding_dim)

        # Initialize embeddings
        nn.init.normal_(self.users_emb.weight, std=0.1)
        nn.init.normal_(self.items_emb.weight, std=0.1)
        
        # Reducing dimensionality to balance
        self.dim_txt_reduction = nn.Linear(self.text_embeddings.shape[1], self.embedding_dim)
        self.dim_img_reduction = nn.Linear(self.image_embeddings.shape[1], self.embedding_dim)
        #self.text_embeddings = nn.Parameter(self.dim_txt_reduction(self.text_embeddings))
        #self.image_embeddings = nn.Parameter(self.dim_img_reduction(self.image_embeddings))

       # Apply dimensionality reduction without making them trainable
        with torch.no_grad():
            #reduced_text = self.dim_txt_reduction(self.text_embeddings)
            reduced_image = self.dim_img_reduction(self.image_embeddings)

        #self.text_embeddings = nn.Parameter(reduced_text, requires_grad=False)
        self.image_embeddings = nn.Parameter(reduced_image, requires_grad=False)

        # Concatenate text and image embeddings without making them trainable
        self.users_embeddings = nn.Parameter(self.image_embeddings[:self.num_users], requires_grad=False)
        self.items_embeddings = nn.Parameter(self.image_embeddings[self.num_users:], requires_grad=False)
    
    

    def forward(self, edge_index: SparseTensor):
        # Compute \tilde{A}: symmetrically normalized adjacency matrix
        edge_index_norm = gcn_norm(edge_index, add_self_loops=self.add_self_loops)

        # Initial embeddings, possibly reduced in dimension
        emb_0 = torch.cat([self.users_emb.weight, self.items_emb.weight])
        embs = [emb_0]
        emb_k = emb_0

        # Propagate embeddings
        for i in range(self.K):
            emb_k = self.propagate(edge_index_norm, x=emb_k)
            embs.append(emb_k)

        embs = torch.stack(embs, dim=1)
        emb_final = torch.mean(embs, dim=1)

        users_emb_final, items_emb_final = torch.split(emb_final, [self.num_users, self.num_items])

        return users_emb_final, self.users_emb.weight, items_emb_final, self.items_emb.weight
  

    def message(self, x_j: torch.Tensor) -> torch.Tensor:
        return x_j

    def message_and_aggregate(self, adj_t: SparseTensor, x: torch.Tensor) -> torch.Tensor:
        # Computes \tilde{A} @ x
        return matmul(adj_t, x)


## Training

In [ ]:
# Initialize the model
model = LightGCN_image_noID(num_users=num_users, num_items=num_movies, text_embeddings=text_embeddings, image_embeddings=image_embeddings, embedding_dim=64)

In [ ]:
# define contants
ITERATIONS = 1000
BATCH_SIZE = 1024
LR = 1e-3
ITERS_PER_EVAL = 50
ITERS_PER_LR_DECAY = 50
K = 20
LAMBDA = 1e-6

In [ ]:
# setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device {device}.")


model = model.to(device)
model.train()

optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.95)

edge_index = edge_index.to(device)
train_edge_index = train_edge_index.to(device)
train_sparse_edge_index = train_sparse_edge_index.to(device)

val_edge_index = val_edge_index.to(device)
val_sparse_edge_index = val_sparse_edge_index.to(device)

In [ ]:
# training loop
train_losses = []
val_losses = []
val_precision = []
val_recall = []
val_ndcg = []
val_srdp = []
val_nov = []


for iter in range(ITERATIONS):
 # forward propagation
    users_emb_final, users_emb_0, items_emb_final, items_emb_0 = model.forward(
        train_sparse_edge_index)

    # mini batching
    user_indices, pos_item_indices, neg_item_indices = sample_mini_batch(
        BATCH_SIZE, train_edge_index)
    user_indices, pos_item_indices, neg_item_indices = user_indices.to(
        device), pos_item_indices.to(device), neg_item_indices.to(device)
    users_emb_final, users_emb_0 = users_emb_final[user_indices], users_emb_0[user_indices]
    pos_items_emb_final, pos_items_emb_0 = items_emb_final[
        pos_item_indices], items_emb_0[pos_item_indices]
    neg_items_emb_final, neg_items_emb_0 = items_emb_final[
        neg_item_indices], items_emb_0[neg_item_indices]

    # loss computation
    train_loss = bpr_loss(users_emb_final, users_emb_0, pos_items_emb_final,
                          pos_items_emb_0, neg_items_emb_final, neg_items_emb_0, LAMBDA)

    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()

    if iter % ITERS_PER_EVAL == 0:
        model.eval()
        val_loss, recall, precision, ndcg, srdp, nov = evaluation(
                model, val_edge_index, val_sparse_edge_index, [train_edge_index], K, LAMBDA)

        print(f"[Iteration {iter}/{ITERATIONS}] train_loss: {round(train_loss.item(), 5)}, val_loss: {round(val_loss, 5)}, val_recall@{K}: {round(recall, 5)}, val_precision@{K}: {round(precision, 5)}, val_ndcg@{K}: {round(ndcg, 5)}, val_srdp@{K}: {round(srdp, 5)}, val_nov@{K}: {round(nov, 5)}")
        train_losses.append(train_loss.item())
        val_losses.append(val_loss)
        val_precision.append(precision)
        val_recall.append(recall)
        val_ndcg.append(ndcg)
        val_srdp.append(srdp)
        val_nov .append(nov)
        model.train()

    if iter % ITERS_PER_LR_DECAY == 0 and iter != 0:
        scheduler.step()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(train_losses))]
plt.plot(iters, train_losses, label='train')
plt.plot(iters, val_losses, label='validation')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('training and validation loss curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_srdp))]
plt.plot(iters, val_srdp, label='srdp')
plt.plot(iters, val_recall, label='recall')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('srdp and recall curves')
plt.legend()
plt.show()

In [ ]:
iters = [iter * ITERS_PER_EVAL for iter in range(len(val_srdp))]
plt.plot(iters, val_ndcg, label='ndcg')
plt.xlabel('iteration')
plt.ylabel('loss')
plt.title('ndcg curve')
plt.legend()
plt.show()

In [ ]:
# evaluate on test set
model.eval()
test_edge_index = test_edge_index.to(device)
test_sparse_edge_index = test_sparse_edge_index.to(device)

test_loss, test_recall, test_precision, test_ndcg, test_srdp, test_nov = evaluation(
            model, test_edge_index, test_sparse_edge_index, [train_edge_index, val_edge_index], K, LAMBDA)

print(f"[test_loss: {round(test_loss, 5)}, test_recall@{K}: {round(test_recall, 5)}, test_precision@{K}: {round(test_precision, 5)}, test_ndcg@{K}: {round(test_ndcg, 5)}, test_srdp@{K}: {round(test_srdp, 5)}, test_nov@{K}: {round(test_nov, 5)}")
print(f"{round(test_loss, 5)}, {round(test_recall, 5)},{round(test_precision, 5)},{round(test_ndcg, 5)},{round(test_srdp, 5)},{round(test_nov, 5)}")